# AI-Powered Research Paper Summarization and Question Answering System (LLM + RAG)

**MCA project – single-notebook edition.** Upload a research paper (PDF), get four kinds of summaries, extract
structured key information, and ask questions that are answered **only from the paper**, with page citations.

### How to run
1. **Run all cells top to bottom** (Colab: *Runtime → Run all*; Jupyter: *Kernel → Restart & Run All*).
2. The last cell launches a **web interface** (Gradio) with tabs: Home, Upload, Summary, Key Information, Ask Document, Evaluation, History.
3. **No API key? No problem.** Without a key the notebook runs in *extractive mode* (no LLM; answers and summaries are built from the most relevant sentences).
   For full LLM quality set `GEMINI_API_KEY` (a free key from https://aistudio.google.com/apikey) in the configuration cell -- it is
   used by default. `LLM_API_KEY` (OpenAI or compatible) and a local Hugging Face model are also supported; see Section 5.

### Notebook map (mirrors the project phases)
| Section | What it contains |
|---|---|
| 1 | Install + configuration + logging + error type |
| 2 | SQLite database |
| 3 | Text cleaning, PDF extraction (`DocumentProcessor`), chunking |
| 4 | Embeddings and vector store (FAISS) |
| 5 | `LLMService` (swappable providers) |
| 6 | Summarization, key-information extraction |
| 7 | Retrieval, RAG question answering, out-of-context detection, conversation memory |
| 8 | Evaluation (ROUGE, BERTScore, Hit@K, precision, recall) |
| 9 | Pipeline wiring, sample paper, self-tests |
| 10 | Web interface |

**Architecture:** `PDF → text extraction → cleaning → chunking → embeddings → FAISS → retrieval → LLM → answer + sources`

## 1. Setup

In [ ]:
%pip install -q pymupdf numpy sentence-transformers faiss-cpu openai requests rouge-score bert-score "gradio>=5,<6" matplotlib pillow python-dotenv

In [ ]:
import os, re, io, json, time, uuid, zlib, shutil, sqlite3, logging, pickle
from collections import Counter
from contextlib import closing
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np

try:                                    # FAISS is preferred; a NumPy fallback keeps the notebook working without it
    import faiss
except ImportError:
    faiss = None

try:                                    # optional: load LLM_API_KEY etc. from a .env file
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


# Gemini models to try, in order. "-latest" aliases are maintained by Google to always point at the
# current recommended model, so they stay correct even as dated model names change; concrete dated
# names are listed after them as a safety net. Flash variants come first ("use flash for speed");
# the "-pro-latest" alias is the final, slower fallback so a question still gets answered if every
# flash model is busy. Override with GEMINI_MODELS="model-a,model-b,..." to use your own list/order.
DEFAULT_GEMINI_MODELS = ["gemini-flash-latest", "gemini-flash-lite-latest", "gemini-2.5-flash",
                         "gemini-2.0-flash", "gemini-pro-latest"]


@dataclass
class Config:
    """All tunable settings. Override with environment variables or edit the defaults below."""
    # ---- LLM: "auto" = Gemini if GEMINI_API_KEY is set, else OpenAI if LLM_API_KEY is set, else extractive mode ----
    llm_provider: str = os.getenv("LLM_PROVIDER", "auto")          # auto | gemini | openai | huggingface | extractive
    llm_api_key: str = os.getenv("LLM_API_KEY", "")                # NEVER hard-code or share a key. Use env var / .env
    llm_model: str = os.getenv("LLM_MODEL", "gpt-4o-mini")
    llm_base_url: str = os.getenv("LLM_BASE_URL", "")              # for OpenAI-compatible providers (Groq, Together, Ollama...)
    llm_timeout_seconds: int = int(os.getenv("LLM_TIMEOUT_SECONDS", "60"))
    hf_model: str = os.getenv("HF_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")  # used when provider = huggingface
    llm_context_tokens: int = int(os.getenv("LLM_CONTEXT_TOKENS", "6000"))  # prompt budget we allow ourselves
    llm_max_output_tokens: int = 900

    # ---- Gemini (Google AI Studio). Get a free key at https://aistudio.google.com/apikey ----
    gemini_api_key: str = os.getenv("GEMINI_API_KEY", "")          # NEVER hard-code or share a key. Use env var / .env
    gemini_models: str = os.getenv("GEMINI_MODELS", ",".join(DEFAULT_GEMINI_MODELS))  # comma-separated, tried in order
    gemini_timeout_seconds: float = float(os.getenv("GEMINI_TIMEOUT_SECONDS", "3.5"))  # per-model timeout before switching
    gemini_discover_fallback: bool = os.getenv("GEMINI_DISCOVER_FALLBACK", "1") == "1"  # auto-discover a working model if all listed ones 404

    # ---- Embeddings ----
    embedding_model: str = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
    embedding_backend: str = os.getenv("EMBEDDING_BACKEND", "auto")  # auto | sentence-transformers | hashing

    # ---- Chunking (in approximate tokens; 1 token ~ 0.75 words) ----
    chunk_size: int = int(os.getenv("CHUNK_SIZE", "500"))
    chunk_overlap: int = int(os.getenv("CHUNK_OVERLAP", "100"))

    # ---- Retrieval ----
    top_k: int = int(os.getenv("TOP_K", "5"))
    relevance_threshold: float = float(os.getenv("RELEVANCE_THRESHOLD", "0.22"))  # below this => "not in document"

    # ---- Upload limits / storage ----
    max_upload_mb: int = int(os.getenv("MAX_UPLOAD_MB", "25"))
    max_pages: int = int(os.getenv("MAX_PAGES", "300"))
    enable_ocr: bool = os.getenv("ENABLE_OCR", "0") == "1"          # optional; needs Tesseract installed
    data_dir: str = os.getenv("DATA_DIR", "rag_project_data")

    def __post_init__(self):
        if self.chunk_overlap >= self.chunk_size:
            raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE")
        if self.chunk_size * 2 > self.llm_context_tokens:
            raise ValueError("CHUNK_SIZE is too large for the LLM context budget")

    @property
    def upload_dir(self): return Path(self.data_dir) / "uploads"
    @property
    def vector_dir(self): return Path(self.data_dir) / "vector_store"
    @property
    def export_dir(self): return Path(self.data_dir) / "exports"
    @property
    def db_path(self): return Path(self.data_dir) / "app.db"


cfg = Config()
for _d in (cfg.upload_dir, cfg.vector_dir, cfg.export_dir):
    _d.mkdir(parents=True, exist_ok=True)

NOT_FOUND = "Not found in document"
NOT_FOUND_ANSWER = "The answer could not be found in the uploaded document."


def estimate_tokens(text: str) -> int:
    """Cheap token estimate (no tokenizer download needed): ~1.3 tokens per word."""
    return int(len(text.split()) * 1.3)


# ---- Logging: console + file (never shown to end users) ----
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s", force=True)
_file_handler = logging.FileHandler(Path(cfg.data_dir) / "app.log")
_file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(name)s | %(message)s"))
logging.getLogger().addHandler(_file_handler)
logger = logging.getLogger("rag")


class AppError(Exception):
    """An error whose message is safe and friendly to show to the user (no keys, no stack traces)."""
    def __init__(self, user_message: str, detail: str = ""):
        super().__init__(user_message)
        self.user_message = user_message
        self.detail = detail


def user_message(exc: Exception) -> str:
    """Convert any exception to a safe message; the real error goes to the log only."""
    if isinstance(exc, AppError):
        logger.warning("AppError: %s | %s", exc.user_message, exc.detail)
        return exc.user_message
    logger.exception("Unexpected error")
    return "Something went wrong. Please try again."


print(f"Data folder: {Path(cfg.data_dir).resolve()}")
print(f"FAISS available: {faiss is not None}")

## 2. Database (SQLite)
Stores documents, page text, cached summaries, extracted information and chat history.

In [ ]:
class Database:
    def __init__(self, path):
        self.path = str(path)
        self._init_schema()

    def _conn(self):
        conn = sqlite3.connect(self.path, check_same_thread=False)
        conn.row_factory = sqlite3.Row
        conn.execute("PRAGMA foreign_keys = ON")
        return conn

    def _run(self, sql, params=(), fetch=None):
        with closing(self._conn()) as conn:
            cur = conn.execute(sql, params)
            rows = None
            if fetch == "one":
                r = cur.fetchone(); rows = dict(r) if r else None
            elif fetch == "all":
                rows = [dict(r) for r in cur.fetchall()]
            conn.commit()
            return rows

    def _init_schema(self):
        with closing(self._conn()) as conn:
            conn.executescript("""
            CREATE TABLE IF NOT EXISTS documents(
                document_id TEXT PRIMARY KEY, filename TEXT NOT NULL, upload_date TEXT NOT NULL,
                page_count INTEGER DEFAULT 0, word_count INTEGER DEFAULT 0,
                processing_status TEXT DEFAULT 'uploaded', error_message TEXT DEFAULT '');
            CREATE TABLE IF NOT EXISTS pages(
                document_id TEXT, page_number INTEGER, text TEXT, has_text INTEGER,
                PRIMARY KEY(document_id, page_number),
                FOREIGN KEY(document_id) REFERENCES documents(document_id) ON DELETE CASCADE);
            CREATE TABLE IF NOT EXISTS summaries(
                document_id TEXT, kind TEXT, content TEXT, created TEXT,
                PRIMARY KEY(document_id, kind),
                FOREIGN KEY(document_id) REFERENCES documents(document_id) ON DELETE CASCADE);
            CREATE TABLE IF NOT EXISTS extractions(
                document_id TEXT PRIMARY KEY, content TEXT, created TEXT,
                FOREIGN KEY(document_id) REFERENCES documents(document_id) ON DELETE CASCADE);
            CREATE TABLE IF NOT EXISTS chat_history(
                id INTEGER PRIMARY KEY AUTOINCREMENT, document_id TEXT, role TEXT, content TEXT,
                sources TEXT, created TEXT,
                FOREIGN KEY(document_id) REFERENCES documents(document_id) ON DELETE CASCADE);
            """)
            conn.commit()

    @staticmethod
    def _now(): return datetime.now().isoformat(timespec="seconds")

    # documents
    def create_document(self, document_id, filename):
        self._run("INSERT INTO documents(document_id, filename, upload_date) VALUES(?,?,?)",
                  (document_id, filename, self._now()))
    def update_status(self, document_id, status, error=""):
        self._run("UPDATE documents SET processing_status=?, error_message=? WHERE document_id=?", (status, error, document_id))
    def update_counts(self, document_id, page_count, word_count):
        self._run("UPDATE documents SET page_count=?, word_count=? WHERE document_id=?", (page_count, word_count, document_id))
    def get_document(self, document_id):
        return self._run("SELECT * FROM documents WHERE document_id=?", (document_id,), "one")
    def list_documents(self):
        return self._run("SELECT * FROM documents ORDER BY upload_date DESC", fetch="all")
    def delete_document(self, document_id):
        self._run("DELETE FROM documents WHERE document_id=?", (document_id,))

    # pages
    def save_pages(self, document_id, pages):
        with closing(self._conn()) as conn:
            conn.executemany("INSERT OR REPLACE INTO pages VALUES(?,?,?,?)",
                             [(document_id, p["page_number"], p["text"], int(p["has_text"])) for p in pages])
            conn.commit()
    def get_pages(self, document_id):
        rows = self._run("SELECT * FROM pages WHERE document_id=? ORDER BY page_number", (document_id,), "all")
        return [{"document_id": r["document_id"], "page_number": r["page_number"], "text": r["text"], "has_text": bool(r["has_text"])} for r in rows]

    # cached summaries / extraction
    def save_summary(self, document_id, kind, content):
        self._run("INSERT OR REPLACE INTO summaries VALUES(?,?,?,?)", (document_id, kind, content, self._now()))
    def get_summary(self, document_id, kind):
        r = self._run("SELECT content FROM summaries WHERE document_id=? AND kind=?", (document_id, kind), "one")
        return r["content"] if r else None
    def save_extraction(self, document_id, data):
        self._run("INSERT OR REPLACE INTO extractions VALUES(?,?,?)", (document_id, json.dumps(data), self._now()))
    def get_extraction(self, document_id):
        r = self._run("SELECT content FROM extractions WHERE document_id=?", (document_id,), "one")
        return json.loads(r["content"]) if r else None

    # chat
    def add_chat(self, document_id, role, content, sources=None):
        self._run("INSERT INTO chat_history(document_id, role, content, sources, created) VALUES(?,?,?,?,?)",
                  (document_id, role, content, json.dumps(sources or []), self._now()))
    def get_chat(self, document_id, limit=None):
        rows = self._run("SELECT role, content, sources, created FROM chat_history WHERE document_id=? ORDER BY id", (document_id,), "all")
        rows = rows[-limit:] if limit else rows
        for r in rows: r["sources"] = json.loads(r["sources"] or "[]")
        return rows
    def clear_chat(self, document_id):
        self._run("DELETE FROM chat_history WHERE document_id=?", (document_id,))


db = Database(cfg.db_path)
print("Database ready:", cfg.db_path)

## 3. Text cleaning, PDF extraction, chunking

In [ ]:
LIGATURES = {"\ufb01": "fi", "\ufb02": "fl", "\ufb00": "ff", "\ufb03": "ffi", "\ufb04": "ffl",
             "\u00a0": " ", "\u2009": " ", "\u200b": "", "\u00ad": ""}


class TextCleaner:
    """Light-touch cleaning: keeps terminology, equations, headings and references intact."""
    PAGE_NUMBER_RE = re.compile(r"^\s*(?:page\s+)?\d{1,3}(?:\s+of\s+\d{1,3})?\s*$", re.I)

    def clean(self, text: str) -> str:
        if not text:
            return ""
        for src, dst in LIGATURES.items():
            text = text.replace(src, dst)
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)            # control characters
        text = re.sub(r"(\w)-\n([a-z])", r"\1\2", text)                       # re-join words hyphenated at line end
        lines = []
        for line in text.split("\n"):
            line = re.sub(r"[ \t]{2,}", " ", line.replace("\t", " ")).strip()  # repeated spaces
            if self.PAGE_NUMBER_RE.match(line):                                 # bare page-number lines
                continue
            lines.append(line)
        text = re.sub(r"\n{3,}", "\n\n", "\n".join(lines))                    # excessive blank lines
        return text.strip()


class DocumentProcessor:
    """Validates uploads and extracts text page by page (page numbers are kept for citations)."""

    def __init__(self, cfg: Config, cleaner: TextCleaner):
        self.cfg, self.cleaner = cfg, cleaner

    def validate_file(self, path, filename=None):
        name = filename or Path(path).name
        if not name.lower().endswith(".pdf"):
            raise AppError("Unsupported file format. Please upload a PDF file.")
        p = Path(path)
        if not p.exists():
            raise AppError("The uploaded file could not be found. Please upload it again.")
        size = p.stat().st_size
        if size == 0:
            raise AppError("The uploaded file is empty.")
        if size > self.cfg.max_upload_mb * 1024 * 1024:
            raise AppError(f"The file is too large. The maximum allowed size is {self.cfg.max_upload_mb} MB.")
        with open(p, "rb") as f:
            if b"%PDF" not in f.read(1024):
                raise AppError("This file is not a valid PDF. It may be corrupted or have the wrong extension.")

    def save_upload(self, path, document_id) -> Path:
        dest = self.cfg.upload_dir / f"{document_id}.pdf"   # random name; the file is stored, never executed
        shutil.copyfile(path, dest)
        return dest

    def _ocr_page(self, page) -> str:
        try:
            tp = page.get_textpage_ocr(language="eng", dpi=200, full=True)
            return page.get_text("text", textpage=tp)
        except Exception as e:
            logger.warning("OCR failed: %s", e)
            return ""

    def extract_pages(self, pdf_path, document_id):
        """Returns [{'document_id', 'page_number', 'text', 'has_text'}, ...]"""
        try:
            import fitz  # PyMuPDF
        except ImportError:
            raise AppError("PyMuPDF is not installed. Run the install cell first.")
        try:
            doc = fitz.open(str(pdf_path))
        except Exception as e:
            raise AppError("The PDF could not be opened. It may be corrupted.", detail=str(e))
        try:
            if getattr(doc, "needs_pass", False):
                raise AppError("This PDF is password-protected. Please upload an unlocked copy.")
            if doc.page_count == 0:
                raise AppError("The PDF has no pages.")
            if doc.page_count > self.cfg.max_pages:
                raise AppError(f"The PDF has {doc.page_count} pages; the limit is {self.cfg.max_pages}.")
            pages = []
            for i in range(doc.page_count):
                try:
                    raw = doc[i].get_text("text")
                except Exception as e:
                    logger.warning("Page %s could not be read: %s", i + 1, e)
                    raw = ""
                text = self.cleaner.clean(raw)
                if not text and self.cfg.enable_ocr:
                    text = self.cleaner.clean(self._ocr_page(doc[i]))
                pages.append({"document_id": document_id, "page_number": i + 1, "text": text, "has_text": bool(text.strip())})
        finally:
            doc.close()
        if not any(p["has_text"] for p in pages):
            raise AppError("No readable text was found. The PDF may be a scanned image (set ENABLE_OCR=1 to try OCR).")
        return pages


# ---------------- Section detection + chunking ----------------
SECTION_PATTERNS = [
    ("Abstract", r"abstract"),
    ("Introduction", r"introduction"),
    ("Related Work", r"related work|literature review|background"),
    ("Methodology", r"methodology|methods?|proposed (?:method|approach|system)|approach|materials and methods"),
    ("Dataset", r"data ?sets?|data|dataset description|data collection"),
    ("Experiments", r"experiments?|experimental (?:setup|results)|evaluation|implementation"),
    ("Results", r"results?(?: and discussion)?|findings"),
    ("Discussion", r"discussion"),
    ("Limitations", r"limitations?|threats to validity"),
    ("Conclusion", r"conclusions?(?: and future work)?|future work|concluding remarks"),
    ("References", r"references|bibliography"),
    ("Acknowledgements", r"acknowledge?ments?"),
]
_SECTION_RES = [(name, re.compile(rf"^(?:(?:\d+(?:\.\d+)*|[IVX]+)[.)]?\s+)?(?:{pat})\s*[:.]?$", re.I))
                for name, pat in SECTION_PATTERNS]


def detect_heading(line: str):
    s = line.strip()
    if not s or len(s) > 70:
        return None
    for name, rx in _SECTION_RES:
        if rx.match(s):
            return name
    return None


def page_label(chunk: dict) -> str:
    a, b = chunk["page_number"], chunk.get("page_end", chunk["page_number"])
    return str(a) if a == b else f"{a}-{b}"


class ChunkingService:
    """Sliding-window chunks that remember the page(s) and section they came from."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.words_per_chunk = max(50, int(cfg.chunk_size / 1.3))
        self.overlap_words = int(cfg.chunk_overlap / 1.3)

    def chunk_pages(self, document_id, pages):
        words = []                               # (word, page_number, section)
        section = "Front matter"
        for p in pages:
            for line in p["text"].split("\n"):
                found = detect_heading(line)
                if found:
                    section = found
                for w in line.split():
                    words.append((w, p["page_number"], section))
        chunks, n = [], len(words)
        step = max(1, self.words_per_chunk - self.overlap_words)
        i, idx = 0, 0
        while i < n:
            window = words[i:i + self.words_per_chunk]
            chunks.append({
                "chunk_id": f"{document_id[:8]}-c{idx:04d}",
                "chunk_index": idx,
                "document_id": document_id,
                "page_number": window[0][1],
                "page_end": window[-1][1],
                "section": Counter(w[2] for w in window).most_common(1)[0][0],
                "text": " ".join(w[0] for w in window),
                "overlap_words": self.overlap_words if idx > 0 else 0,
            })
            idx += 1
            if i + self.words_per_chunk >= n:
                break
            i += step
        return chunks


cleaner = TextCleaner()
processor = DocumentProcessor(cfg, cleaner)
chunker = ChunkingService(cfg)
print(f"Chunking: ~{chunker.words_per_chunk} words per chunk, ~{chunker.overlap_words} words overlap")

## 4. Embeddings and vector database
* `EmbeddingService` uses **all-MiniLM-L6-v2** (configurable). If sentence-transformers cannot be loaded (e.g. offline), it falls back to a
  lightweight hashed bag-of-words embedding so the notebook keeps working.
* `VectorStore` uses **FAISS** (inner product on normalised vectors = cosine similarity) and can save/load itself.

In [ ]:
_STOPWORDS = set("a an the of and or to in on for with by is are was were be been this that these those it its as at from we our using "
                 "use used which can also than then their they them has have had not but into over such what who how when where why do does did".split())


class EmbeddingService:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.model = None
        self.backend = None

    def _load(self):
        if self.backend:
            return
        if self.cfg.embedding_backend in ("auto", "sentence-transformers"):
            try:
                from sentence_transformers import SentenceTransformer
                self.model = SentenceTransformer(self.cfg.embedding_model)
                self.backend = "sentence-transformers"
                logger.info("Loaded embedding model %s", self.cfg.embedding_model)
                return
            except Exception as e:
                if self.cfg.embedding_backend == "sentence-transformers":
                    raise AppError("The embedding model could not be loaded. Check your internet connection and the model name.", detail=str(e))
                logger.warning("sentence-transformers unavailable (%s); using hashing embeddings", e)
        self.backend = "hashing"

    @property
    def embedding_id(self) -> str:
        self._load()
        return f"{self.backend}:{self.cfg.embedding_model if self.backend != 'hashing' else 'hashing-1024'}"

    @staticmethod
    def _hash_embed(text: str, dim: int = 1024) -> np.ndarray:
        toks = [t for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in _STOPWORDS]
        feats = toks + [a + "_" + b for a, b in zip(toks, toks[1:])]
        v = np.zeros(dim, dtype=np.float32)
        for f in feats:
            v[zlib.crc32(f.encode()) % dim] += 1.0
        return np.log1p(v)

    def encode(self, texts, batch_size: int = 32) -> np.ndarray:
        """Returns an (n, dim) float32 array of L2-normalised vectors."""
        self._load()
        texts = [t if t.strip() else " " for t in texts]
        try:
            if self.backend == "sentence-transformers":
                emb = self.model.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False)
            else:
                emb = np.vstack([self._hash_embed(t) for t in texts])
        except Exception as e:
            raise AppError("Embedding generation failed. Please try again.", detail=str(e))
        emb = np.asarray(emb, dtype=np.float32)
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        return emb / np.where(norms == 0, 1, norms)


class VectorStore:
    """chunk text + metadata + embedding, searchable by cosine similarity."""

    def __init__(self, embedder: EmbeddingService, store_dir):
        self.embedder = embedder
        self.dir = Path(store_dir)
        self.chunks: list = []
        self.embeddings = None
        self.index = None
        self.stored_embedding_id = None

    def _rebuild_index(self):
        self.index = None
        if faiss is not None and self.embeddings is not None and len(self.chunks):
            self.index = faiss.IndexFlatIP(self.embeddings.shape[1])
            self.index.add(np.ascontiguousarray(self.embeddings, dtype="float32"))

    def _check_compat(self):
        if self.stored_embedding_id and self.stored_embedding_id != self.embedder.embedding_id:
            raise AppError("The saved vectors were created with a different embedding model. "
                           "Delete the documents (History tab) and upload them again.")

    def add_embeddings(self, chunks, embeddings):
        if len(chunks) != len(embeddings):
            raise AppError("Internal error: chunks and embeddings do not match.")
        self._check_compat()
        try:
            embeddings = np.asarray(embeddings, dtype=np.float32)
            self.embeddings = embeddings if self.embeddings is None else np.vstack([self.embeddings, embeddings])
            self.chunks.extend(dict(c) for c in chunks)
            self.stored_embedding_id = self.embedder.embedding_id
            self._rebuild_index()
        except AppError:
            raise
        except Exception as e:
            raise AppError("The vector database could not store the document.", detail=str(e))

    def add_documents(self, chunks):
        self.add_embeddings(chunks, self.embedder.encode([c["text"] for c in chunks]))

    def search(self, query: str, top_k: int = 5, document_id: str = None):
        """Returns the top-k chunks (text, page, section, score, document_id)."""
        if not self.chunks:
            return []
        self._check_compat()
        q = self.embedder.encode([query])
        n = len(self.chunks)
        try:
            if self.index is not None:
                scores, idx = self.index.search(np.ascontiguousarray(q), n)
                pairs = zip(idx[0], scores[0])
            else:
                all_scores = self.embeddings @ q[0]
                pairs = [(int(i), float(all_scores[i])) for i in np.argsort(-all_scores)]
            out = []
            for i, s in pairs:
                if i < 0:
                    continue
                c = self.chunks[int(i)]
                if document_id and c["document_id"] != document_id:
                    continue
                out.append({**c, "score": float(s)})
                if len(out) >= top_k:
                    break
            return out
        except Exception as e:
            raise AppError("The vector search failed. Please try again.", detail=str(e))

    def get_chunks(self, document_id):
        return sorted((c for c in self.chunks if c["document_id"] == document_id), key=lambda c: c["chunk_index"])

    def has_document(self, document_id) -> bool:
        return any(c["document_id"] == document_id for c in self.chunks)

    def delete_document(self, document_id):
        keep = [i for i, c in enumerate(self.chunks) if c["document_id"] != document_id]
        self.chunks = [self.chunks[i] for i in keep]
        self.embeddings = self.embeddings[keep] if self.embeddings is not None and keep else None
        if not self.chunks:
            self.stored_embedding_id = None
        self._rebuild_index()

    def clear(self):
        self.chunks, self.embeddings, self.index, self.stored_embedding_id = [], None, None, None

    def save(self):
        self.dir.mkdir(parents=True, exist_ok=True)
        if self.embeddings is None or not self.chunks:
            for f in ("chunks.json", "embeddings.npy"):
                (self.dir / f).unlink(missing_ok=True)
            return
        np.save(self.dir / "embeddings.npy", self.embeddings)
        with open(self.dir / "chunks.json", "w", encoding="utf-8") as f:
            json.dump({"embedding_id": self.stored_embedding_id, "chunks": self.chunks}, f)

    def load(self):
        cj, en = self.dir / "chunks.json", self.dir / "embeddings.npy"
        if not (cj.exists() and en.exists()):
            return False
        try:
            data = json.loads(cj.read_text(encoding="utf-8"))
            self.chunks, self.stored_embedding_id = data["chunks"], data.get("embedding_id")
            self.embeddings = np.load(en)
            self._rebuild_index()          # the FAISS index is cheap to rebuild, so only vectors + metadata are stored
            return True
        except Exception as e:
            logger.warning("Could not load vector store: %s", e)
            self.clear()
            return False


embedder = EmbeddingService(cfg)


def effective_threshold(threshold: float) -> float:
    """Similarity scores from the offline hashing fallback are much lower than MiniLM scores, so scale the threshold down."""
    embedder._load()
    return threshold * 0.2 if embedder.backend == "hashing" else threshold


vstore = VectorStore(embedder, cfg.vector_dir)
print("Vector store loaded from disk:", vstore.load(), f"({len(vstore.chunks)} chunks)")

## 5. LLM service (swappable providers)
`LLMService` hides the provider. Choose it with `LLM_PROVIDER`:
* `gemini` – Google Gemini (set `GEMINI_API_KEY`). Tries several models in order -- Flash first, for speed --
  and automatically switches to the next model if one is slow (> `GEMINI_TIMEOUT_SECONDS`, default 3.5s),
  rate-limited, or overloaded, so a busy model does not stall the whole app. See `GeminiProvider` below.
* `openai` – OpenAI or any OpenAI-compatible endpoint (set `LLM_BASE_URL`)
* `huggingface` – a local Hugging Face chat model (`HF_MODEL`)
* `extractive` – no LLM at all (offline demo mode)
* `auto` (default) – Gemini if `GEMINI_API_KEY` is set, else OpenAI if `LLM_API_KEY` is set, else extractive

In [ ]:
class LLMProvider:
    name = "base"
    generative = True

    def __init__(self, cfg=None):
        self.cfg = cfg

    def generate(self, system: str, user: str, max_tokens: int = 512, temperature: float = 0.1) -> str:
        raise NotImplementedError


class OpenAIProvider(LLMProvider):
    def __init__(self, cfg: Config):
        self.cfg, self._client = cfg, None
        self.name = f"openai:{cfg.llm_model}"

    def _get_client(self):
        if self._client is None:
            if not self.cfg.llm_api_key:
                raise AppError("No LLM API key is configured. Set LLM_API_KEY in the configuration cell, or use extractive mode.")
            try:
                from openai import OpenAI
            except ImportError:
                raise AppError("The 'openai' package is not installed. Run the install cell.")
            kwargs = dict(api_key=self.cfg.llm_api_key, timeout=self.cfg.llm_timeout_seconds, max_retries=2)
            if self.cfg.llm_base_url:
                kwargs["base_url"] = self.cfg.llm_base_url
            self._client = OpenAI(**kwargs)
        return self._client

    def generate(self, system, user, max_tokens=512, temperature=0.1):
        client = self._get_client()
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        try:
            try:
                resp = client.chat.completions.create(model=self.cfg.llm_model, messages=messages,
                                                      max_tokens=max_tokens, temperature=temperature)
            except Exception as e:
                if "max_completion_tokens" not in str(e):
                    raise
                resp = client.chat.completions.create(model=self.cfg.llm_model, messages=messages, max_completion_tokens=max_tokens)
            return (resp.choices[0].message.content or "").strip()
        except AppError:
            raise
        except Exception as e:
            kind = type(e).__name__
            logger.error("LLM call failed (%s): %s", kind, e)
            if "Timeout" in kind:
                msg = "The language model took too long to respond. Please try again."
            elif "Authentication" in kind or "PermissionDenied" in kind:
                msg = "The LLM API key was rejected. Please check LLM_API_KEY."
            elif "RateLimit" in kind:
                msg = "The LLM rate limit was reached. Please wait a moment and try again."
            elif "Connection" in kind:
                msg = "Could not reach the LLM service. Check your internet connection."
            else:
                msg = "The language model request failed. Please try again."
            raise AppError(msg, detail=str(e))


class GeminiProvider(LLMProvider):
    """Google Gemini via the REST API (no SDK dependency, so we control the per-call timeout exactly).

    Countermeasures for a busy model, as requested:
    * Each model gets at most `cfg.gemini_timeout_seconds` (default 3.5s) to respond.
    * On a timeout, a rate-limit (429) or an overloaded/server error (503/500), the NEXT model in
      `cfg.gemini_models` is tried immediately -- the flash models are tried first for speed, with
      slower/older models as a safety net.
    * An invalid API key (401/403) stops immediately, since trying other models cannot fix that.
    * If every configured model answers 404 ("model not found"), the model names may be out of date;
      the provider then calls the API's own model list and retries a couple of live "flash" models
      it finds there, so the notebook keeps working even if Google renames or retires a model.
    """
    API_BASE = "https://generativelanguage.googleapis.com/v1beta"

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.models = [m.strip() for m in cfg.gemini_models.split(",") if m.strip()]
        if not self.models:
            raise AppError("No Gemini models are configured. Set GEMINI_MODELS to at least one model name.")
        self.name = f"gemini:{self.models[0]} (+{len(self.models) - 1} fallback)" if len(self.models) > 1 else f"gemini:{self.models[0]}"
        self._last_working_model = None

    def _require_requests(self):
        try:
            import requests
            return requests
        except ImportError:
            raise AppError("The 'requests' package is not installed. Run the install cell.")

    def _call_model(self, requests, model, system, user, max_tokens, temperature):
        """One attempt against one model. Returns text, or raises to signal 'try the next model'."""
        url = f"{self.API_BASE}/models/{model}:generateContent"
        payload = {
            "systemInstruction": {"parts": [{"text": system}]},
            "contents": [{"role": "user", "parts": [{"text": user}]}],
            "generationConfig": {"temperature": temperature, "maxOutputTokens": max_tokens},
        }
        resp = requests.post(url, params={"key": self.cfg.gemini_api_key}, json=payload,
                             timeout=self.cfg.gemini_timeout_seconds)
        if resp.status_code in (401, 403):
            raise AppError("The Gemini API key was rejected. Please check GEMINI_API_KEY.")
        if resp.status_code == 404:
            raise LookupError(f"model '{model}' not found")              # caught by caller, triggers discovery
        if resp.status_code == 429:
            raise TimeoutError(f"model '{model}' is rate-limited")       # caught by caller, tries next model
        if resp.status_code in (500, 503, 504):
            raise TimeoutError(f"model '{model}' is overloaded ({resp.status_code})")
        resp.raise_for_status()
        data = resp.json()
        candidates = data.get("candidates") or []
        if not candidates:
            reason = data.get("promptFeedback", {}).get("blockReason", "no candidates returned")
            raise ValueError(f"model '{model}' returned nothing ({reason})")
        parts = candidates[0].get("content", {}).get("parts", [])
        text = "".join(p.get("text", "") for p in parts).strip()
        if not text:
            raise ValueError(f"model '{model}' returned an empty response")
        return text

    def _discover_models(self, requests, already_tried):
        """Asks the Gemini API which models actually exist for this key, as a last resort."""
        try:
            resp = requests.get(f"{self.API_BASE}/models", params={"key": self.cfg.gemini_api_key},
                                timeout=self.cfg.gemini_timeout_seconds)
            resp.raise_for_status()
            found = []
            for m in resp.json().get("models", []):
                name = m.get("name", "").removeprefix("models/")
                if (name and name not in already_tried and "generateContent" in m.get("supportedGenerationMethods", [])
                        and "flash" in name.lower()):
                    found.append(name)
            return found[:3]                                              # cap how many extra attempts we make
        except Exception as e:
            logger.warning("Gemini model discovery failed: %s", e)
            return []

    def generate(self, system, user, max_tokens=512, temperature=0.1):
        if not self.cfg.gemini_api_key:
            raise AppError("No Gemini API key is configured. Set GEMINI_API_KEY in the configuration cell.")
        requests = self._require_requests()
        tried, saw_unknown_model, last_error = [], False, None
        candidates = ([self._last_working_model] if self._last_working_model else []) +                     [m for m in self.models if m != self._last_working_model]  # retry the last model that worked, first
        extra_round = False
        while candidates:
            for model in candidates:
                tried.append(model)
                try:
                    text = self._call_model(requests, model, system, user, max_tokens, temperature)
                    self._last_working_model = model
                    return text
                except AppError:
                    raise                                                  # bad API key: no point trying more models
                except LookupError as e:
                    saw_unknown_model = True
                    last_error = e
                    logger.warning("Gemini: %s, trying next model", e)
                except (TimeoutError, requests.exceptions.Timeout) as e:
                    last_error = e
                    logger.warning("Gemini: %s after %.1fs, trying next model", e, self.cfg.gemini_timeout_seconds)
                except Exception as e:
                    last_error = e
                    logger.warning("Gemini: model '%s' failed (%s), trying next model", model, e)
            if extra_round or not (saw_unknown_model and self.cfg.gemini_discover_fallback):
                break
            discovered = self._discover_models(requests, set(tried))       # one extra round using live model names
            if not discovered:
                break
            candidates, extra_round = discovered, True
        raise AppError("All Gemini models are currently busy or unavailable. Please try again in a moment.",
                       detail=f"tried {tried}: {last_error}")


class HuggingFaceProvider(LLMProvider):
    def __init__(self, cfg: Config):
        self.cfg, self._pipe = cfg, None
        self.name = f"huggingface:{cfg.hf_model}"

    def _get_pipe(self):
        if self._pipe is None:
            try:
                import torch
                from transformers import pipeline
                device = 0 if torch.cuda.is_available() else -1
                self._pipe = pipeline("text-generation", model=self.cfg.hf_model, device=device)
            except Exception as e:
                raise AppError("The local Hugging Face model could not be loaded.", detail=str(e))
        return self._pipe

    def generate(self, system, user, max_tokens=512, temperature=0.1):
        pipe = self._get_pipe()
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        try:
            kwargs = dict(max_new_tokens=max_tokens, return_full_text=False)
            kwargs.update(dict(do_sample=True, temperature=temperature) if temperature > 0 else dict(do_sample=False))
            out = pipe(messages, **kwargs)[0]["generated_text"]
            if isinstance(out, list):
                out = out[-1]["content"]
            return out.strip()
        except Exception as e:
            raise AppError("The local language model failed to generate a response.", detail=str(e))


class ExtractiveProvider(LLMProvider):
    """Placeholder: signals that no generative model is available."""
    name = "extractive (no LLM)"
    generative = False

    def generate(self, *a, **k):
        raise AppError("No language model is configured.")


class LLMService:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        choice = cfg.llm_provider.lower()
        if choice == "auto":                                 # Gemini first (fast + generous free tier), then OpenAI, then none
            choice = "gemini" if cfg.gemini_api_key else ("openai" if cfg.llm_api_key else "extractive")
        providers = {"gemini": GeminiProvider, "openai": OpenAIProvider, "huggingface": HuggingFaceProvider,
                    "extractive": ExtractiveProvider}
        if choice not in providers:
            raise AppError(f"Unknown LLM provider '{cfg.llm_provider}'. Use auto, gemini, openai, huggingface or extractive.")
        self.provider = providers[choice](cfg)

    @property
    def generative(self) -> bool: return self.provider.generative
    @property
    def name(self) -> str: return self.provider.name

    def generate(self, system: str, user: str, max_tokens: int = None, temperature: float = 0.1) -> str:
        max_tokens = max_tokens or self.cfg.llm_max_output_tokens
        if estimate_tokens(system + user) + max_tokens > self.cfg.llm_context_tokens * 1.15:
            logger.warning("Prompt is larger than the configured context budget (%s tokens)", estimate_tokens(system + user))
        start = time.time()
        text = self.provider.generate(system, user, max_tokens, temperature)
        logger.info("LLM call took %.1fs", time.time() - start)
        return text


llm = LLMService(cfg)
print("LLM provider:", llm.name)

## 6. Summarization and key-information extraction
* **Hierarchical summarization:** chunks → grouped to fit the context window → group summaries → combined (repeated until small) → final summary of the chosen type.
* **Extractive fallback** (no LLM): sentences are ranked by embedding similarity (to the paper's centroid, or to a field question).

In [ ]:
FIELD_QUERIES = {
    "research_problem": "What research problem or gap does this paper address?",
    "objective": "What is the objective, aim or contribution of this study?",
    "methodology": "What methodology or approach was used?",
    "dataset": "What dataset or data was used?",
    "algorithms_models": "Which algorithms or models were used?",
    "experimental_setup": "What was the experimental setup, parameters, train test split and evaluation metrics?",
    "results": "What were the experimental results and performance?",
    "important_findings": "What are the key findings?",
    "limitations": "What are the limitations of the study?",
    "conclusion": "What is the conclusion and future work?",
}
FIELD_SECTIONS = {
    "research_problem": ["Introduction", "Abstract"], "objective": ["Introduction", "Abstract"],
    "methodology": ["Methodology"], "dataset": ["Dataset", "Methodology", "Experiments"],
    "algorithms_models": ["Methodology", "Experiments"], "experimental_setup": ["Experiments", "Methodology"],
    "results": ["Results", "Experiments"], "important_findings": ["Results", "Discussion", "Conclusion"],
    "limitations": ["Limitations", "Discussion", "Conclusion"], "conclusion": ["Conclusion"],
}
SKIP_SECTIONS = ("References", "Acknowledgements")


class ExtractiveEngine:
    """Sentence ranking with embeddings. Used when there is no LLM (and for the extractive answer)."""

    def __init__(self, embedder: EmbeddingService):
        self.embedder = embedder

    @staticmethod
    def split_sentences(text: str):
        parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(\[])", text.replace("\n", " "))
        return [s.strip() for s in parts if 25 <= len(s.strip()) <= 600]

    def sentences_from(self, chunks):
        seen, out = set(), []
        for c in chunks:
            for s in self.split_sentences(c["text"]):
                key = re.sub(r"\W+", "", s.lower())[:80]
                if key in seen:
                    continue
                seen.add(key)
                out.append({"text": s, "page": c["page_number"], "section": c["section"], "order": len(out)})
        return out

    def rank(self, query, chunks, n=3):
        sents = self.sentences_from(chunks)
        if not sents:
            return []
        emb = self.embedder.encode([s["text"] for s in sents])
        q = self.embedder.encode([query])[0]
        scores = emb @ q
        top = np.argsort(-scores)[:n]
        return [{**sents[i], "score": float(scores[i])} for i in top]

    def centroid_summary(self, chunks, n=5):
        sents = self.sentences_from(chunks)
        if not sents:
            return []
        emb = self.embedder.encode([s["text"] for s in sents])
        centroid = emb.mean(axis=0)
        centroid /= (np.linalg.norm(centroid) or 1)
        chosen = []
        for i in np.argsort(-(emb @ centroid)):
            if all(float(emb[i] @ emb[j]) < 0.85 for j in chosen):     # skip near-duplicates
                chosen.append(int(i))
            if len(chosen) >= n:
                break
        return [sents[i] for i in sorted(chosen, key=lambda i: sents[i]["order"])]

    def extract_field(self, field, chunks, threshold, n=2):
        """Best sentences for a field, preferring the sections where that field normally appears."""
        preferred = [c for c in chunks if c["section"] in FIELD_SECTIONS.get(field, [])]
        if preferred:                      # the section name is strong evidence, so no similarity threshold here
            ranked = self.rank(FIELD_QUERIES[field], preferred, n)
            if ranked:
                return " ".join(s["text"] for s in sorted(ranked, key=lambda s: s["order"]))
        ranked = self.rank(FIELD_QUERIES[field], chunks, n)
        if ranked and ranked[0]["score"] >= effective_threshold(threshold):
            return " ".join(s["text"] for s in sorted(ranked, key=lambda s: s["order"]))
        return None


extractive = ExtractiveEngine(embedder)


# ------------------------------ Summarization ------------------------------
SUMMARY_SYSTEM = ("You are an expert academic assistant. Use ONLY the paper content provided. "
                  "Do not invent facts, numbers or citations. If something is not stated, say that it is not stated.")

SUMMARY_PROMPTS = {
    "Brief": ("Write a brief summary of this research paper in 3 to 5 sentences covering the problem, method and main result.", 350),
    "Detailed": ("Write a detailed, well-organised summary of this research paper in 3 to 5 short paragraphs. "
                 "Cover the background, objective, methodology, dataset, results and conclusion, keeping concrete numbers.", 900),
    "Bullet Points": ("Summarise the main points of this research paper as 6 to 10 concise bullet points, each starting with '- '.", 500),
    "Academic Summary": ("Write a structured academic summary using exactly these markdown headings: "
                         "**Research Problem**, **Objective**, **Methodology**, **Dataset**, **Results**, **Findings**, **Conclusion**. "
                         "Under each heading write 1 to 3 sentences. If the paper does not state something, write 'Not found in document'.", 900),
}


class SummarizationService:
    TYPES = list(SUMMARY_PROMPTS)

    def __init__(self, cfg, db, vstore, llm, extractive):
        self.cfg, self.db, self.vstore, self.llm, self.extractive = cfg, db, vstore, llm, extractive

    # ---- helpers ----
    def _budget_words(self):
        return max(300, int((self.cfg.llm_context_tokens - self.cfg.llm_max_output_tokens - 600) / 1.3))

    def _doc_chunks(self, document_id):
        chunks = [c for c in self.vstore.get_chunks(document_id) if c["section"] not in SKIP_SECTIONS]
        if not chunks:
            raise AppError("This document has no content available for summarisation.")
        return chunks

    @staticmethod
    def _without_overlap(chunks):
        """Drop the overlapping words at the start of consecutive chunks so text is not repeated."""
        texts, prev = [], None
        for c in chunks:
            words = c["text"].split()
            if prev is not None and c["chunk_index"] == prev + 1:
                words = words[c.get("overlap_words", 0):]
            texts.append(" ".join(words))
            prev = c["chunk_index"]
        return texts

    def _batch(self, texts, budget_words):
        batches, current, size = [], [], 0
        for t in texts:
            w = t.split()
            if len(w) > budget_words:
                t, w = " ".join(w[:budget_words]), w[:budget_words]
            if current and size + len(w) > budget_words:
                batches.append(current); current, size = [], 0
            current.append(t); size += len(w)
        if current:
            batches.append(current)
        return batches

    def _digest(self, document_id, chunks, force=False, progress=None):
        """Hierarchical (map-reduce) condensation of the whole paper so that it fits the LLM context window."""
        if not force:
            cached = self.db.get_summary(document_id, "_digest")
            if cached:
                return cached
        budget = self._budget_words()
        texts = self._without_overlap(chunks)
        if sum(len(t.split()) for t in texts) <= budget:
            digest = "\n\n".join(texts)                     # short paper: no need to condense
        else:
            level = 1
            while True:
                batches = self._batch(texts, budget)
                new_texts = []
                for k, b in enumerate(batches, 1):
                    if progress:
                        progress(f"Summarising part {k}/{len(batches)} (level {level})...")
                    new_texts.append(self.llm.generate(
                        SUMMARY_SYSTEM,
                        "Summarise this excerpt of a research paper in 5 to 7 sentences. Keep concrete details such as dataset "
                        f"names, methods, parameters and numbers.\n\nExcerpt:\n{chr(10).join(b)}",
                        max_tokens=450))
                if sum(len(t.split()) for t in new_texts) <= budget or len(new_texts) <= 1 or len(new_texts) >= len(texts):
                    texts = new_texts
                    break
                texts, level = new_texts, level + 1
            digest = "\n\n".join(texts)
        self.db.save_summary(document_id, "_digest", digest)
        return digest

    # ---- extractive summaries (no LLM) ----
    def _extractive(self, kind, chunks):
        if kind == "Brief":
            return " ".join(s["text"] for s in self.extractive.centroid_summary(chunks, 4))
        if kind == "Detailed":
            return " ".join(s["text"] for s in self.extractive.centroid_summary(chunks, 12))
        if kind == "Bullet Points":
            return "\n".join("- " + s["text"] for s in self.extractive.centroid_summary(chunks, 8))
        headings = [("Research Problem", "research_problem"), ("Objective", "objective"), ("Methodology", "methodology"),
                    ("Dataset", "dataset"), ("Results", "results"), ("Findings", "important_findings"), ("Conclusion", "conclusion")]
        parts = []
        for title, field in headings:
            text = self.extractive.extract_field(field, chunks, self.cfg.relevance_threshold) or NOT_FOUND
            parts.append(f"**{title}**\n{text}")
        return "\n\n".join(parts)

    # ---- public API ----
    def summarize(self, document_id, kind="Brief", force=False, progress=None):
        if kind not in SUMMARY_PROMPTS:
            raise AppError(f"Unknown summary type '{kind}'.")
        start = time.time()
        if not force:
            cached = self.db.get_summary(document_id, kind)
            if cached:
                return {"kind": kind, "text": cached, "mode": "cached", "seconds": 0.0}
        chunks = self._doc_chunks(document_id)
        if self.llm.generative:
            digest = self._digest(document_id, chunks, force, progress)
            instruction, max_tokens = SUMMARY_PROMPTS[kind]
            text = self.llm.generate(SUMMARY_SYSTEM, f"{instruction}\n\nPaper content:\n{digest}", max_tokens=max_tokens)
            mode = "llm"
        else:
            text, mode = self._extractive(kind, chunks), "extractive"
        if not text.strip():
            raise AppError("The summary came back empty. Please try again.")
        self.db.save_summary(document_id, kind, text)
        return {"kind": kind, "text": text, "mode": mode, "seconds": time.time() - start}


# ------------------------------ Key information ------------------------------
INFO_FIELDS = ["title", "authors", "abstract", "research_problem", "objective", "methodology", "dataset",
               "algorithms_models", "experimental_setup", "results", "important_findings", "limitations",
               "conclusion", "keywords"]
LIST_FIELDS = {"authors", "algorithms_models", "important_findings", "keywords"}


class KeyInformationService:
    def __init__(self, cfg, db, vstore, llm, extractive, retriever):
        self.cfg, self.db, self.vstore, self.llm, self.extractive, self.retriever = cfg, db, vstore, llm, extractive, retriever

    # ---- LLM path ----
    def _build_context(self, document_id):
        budget = int((self.cfg.llm_context_tokens - self.cfg.llm_max_output_tokens - 700) / 1.3)
        parts, used = [], set()
        pages = self.db.get_pages(document_id)
        if pages:                                     # raw first page keeps line breaks: best source for title/authors
            parts.append("[First page]\n" + " \n".join(pages[0]["text"].split("\n")[:60]))
        for field, query in FIELD_QUERIES.items():
            for r in self.retriever.retrieve(document_id, query, top_k=2)["results"]:
                if r["chunk_id"] not in used and r["section"] not in SKIP_SECTIONS:
                    used.add(r["chunk_id"])
                    parts.append(f"[Page {page_label(r)} | {r['section']}]\n{r['text']}")
        words, out = 0, []
        for p in parts:
            w = len(p.split())
            if words + w > budget:
                break
            out.append(p); words += w
        return "\n\n".join(out)

    @staticmethod
    def _parse_json(text):
        text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            a, b = text.find("{"), text.rfind("}")
            if a != -1 and b > a:
                return json.loads(text[a:b + 1])
            raise

    @staticmethod
    def normalize(data: dict) -> dict:
        """Guarantee every field exists; missing or empty values become 'Not found in document'."""
        out = {}
        for f in INFO_FIELDS:
            v = data.get(f) if isinstance(data, dict) else None
            if isinstance(v, str):
                v = v.strip()
            if v in (None, "", [], {}) or (isinstance(v, str) and v.lower() in ("n/a", "none", "not found", "unknown", "not found in document")):
                v = NOT_FOUND
            if isinstance(v, list):
                v = [str(x).strip() for x in v if str(x).strip()] or NOT_FOUND
                if v != NOT_FOUND and v[0].lower() == "not found in document":
                    v = NOT_FOUND
            if f in LIST_FIELDS and isinstance(v, str) and v != NOT_FOUND:
                v = [x.strip() for x in re.split(r"[;,]", v) if x.strip()] or NOT_FOUND
            out[f] = v
        return out

    def _llm_extract(self, document_id):
        template = json.dumps({f: ([] if f in LIST_FIELDS else "...") for f in INFO_FIELDS}, indent=1)
        system = ("You extract structured information from a research paper. Use ONLY the provided text. "
                  f"If a field is not present in the text, use the string \"{NOT_FOUND}\". Never guess. Return valid JSON only.")
        user = (f"Return a JSON object with exactly these keys (lists for authors, algorithms_models, important_findings, keywords; "
                f"strings for the rest, each string at most 3 sentences):\n{template}\n\nPaper text:\n{self._build_context(document_id)}")
        last_error = None
        for _ in range(2):
            raw = self.llm.generate(system, user, max_tokens=self.cfg.llm_max_output_tokens, temperature=0.0)
            try:
                return self.normalize(self._parse_json(raw))
            except Exception as e:
                last_error = e
                user += "\n\nYour previous reply was not valid JSON. Reply with the JSON object only."
        raise AppError("The model did not return valid structured data. Please try again.", detail=str(last_error))

    # ---- extractive path ----
    def _extractive_extract(self, document_id):
        chunks = [c for c in self.vstore.get_chunks(document_id) if c["section"] not in SKIP_SECTIONS]
        pages = self.db.get_pages(document_id)
        first = pages[0]["text"] if pages else ""
        first2 = "\n".join(p["text"] for p in pages[:2])
        lines = [l.strip() for l in first.split("\n")]
        data = {}

        paragraphs = [p for p in re.split(r"\n\s*\n", first) if p.strip()]
        title_lines = []
        if paragraphs and len(paragraphs[0].split()) <= 30:
            title_lines = [l.strip() for l in paragraphs[0].split("\n")]
        else:
            title_lines = [next((l for l in lines if len(l.split()) >= 3), "")]
        title = " ".join(l for l in title_lines if l)
        data["title"] = title or NOT_FOUND

        name = r"[A-Z][\w.'\-]+(?: [A-Z][\w.'\-]+){0,3}"
        author_re = re.compile(rf"^{name}(?:(?:, and |, | and | & ){name})*$")
        after = [l for l in lines if l and l not in title_lines]
        authors = next((l for l in after[:6] if len(l) <= 120 and not re.search(r"\d", l) and author_re.match(l)), None)
        data["authors"] = [a.strip() for a in re.split(r",| and | & ", authors) if a.strip()] if authors else NOT_FOUND

        m = re.search(r"\babstract\b\W*\n?(.{80,1800}?)(?:\n\s*\n|\n\s*(?:1\.?\s*)?introduction\b|$)", first2, re.I | re.S)
        data["abstract"] = re.sub(r"\s+", " ", m.group(1)).strip() if m else NOT_FOUND

        m = re.search(r"keywords?\s*[:\u2014\-]\s*(.+)", first2, re.I)
        data["keywords"] = [k.strip() for k in re.split(r"[;,\u00b7\u2022]", m.group(1)) if k.strip()][:10] if m else NOT_FOUND

        for field in FIELD_QUERIES:
            text = self.extractive.extract_field(field, chunks, self.cfg.relevance_threshold)
            if field in LIST_FIELDS and text:
                text = self.extractive.split_sentences(text) or text
            data[field] = text or NOT_FOUND
        return self.normalize(data)

    def extract(self, document_id, force=False):
        if not force:
            cached = self.db.get_extraction(document_id)
            if cached:
                return cached
        if not self.vstore.has_document(document_id):
            raise AppError("This document has not been processed yet.")
        data = self._llm_extract(document_id) if self.llm.generative else self._extractive_extract(document_id)
        self.db.save_extraction(document_id, data)
        return data

## 7. Retrieval, RAG question answering, out-of-context detection, conversation memory
1. **Follow-up handling:** a follow-up such as *"How many categories does it contain?"* is rewritten into a standalone question using the last exchange only.
2. **Retrieval:** the standalone question is embedded and the top-K chunks are fetched from FAISS.
3. **Out-of-context check:** if the best similarity is below the (configurable) threshold, the system answers *"The answer could not be found in the uploaded document."* without calling the LLM.
4. **Grounded generation:** the LLM only sees the retrieved context plus the standalone question (not the older chat), which prevents old turns from contaminating the answer.

In [ ]:
class RetrievalService:
    def __init__(self, cfg, vstore):
        self.cfg, self.vstore = cfg, vstore

    def retrieve(self, document_id, query, top_k=None, threshold=None):
        start = time.time()
        top_k = top_k or self.cfg.top_k
        threshold = effective_threshold(self.cfg.relevance_threshold if threshold is None else threshold)
        results = self.vstore.search(query, top_k, document_id)
        top = results[0]["score"] if results else 0.0
        return {"results": results, "top_score": top, "threshold": threshold,
                "relevant": bool(results) and top >= threshold, "seconds": time.time() - start}


QA_SYSTEM = ("You answer questions about one research paper. You are given a short paper summary for orientation and a "
             "few specific excerpts for exact facts, figures and names -- prefer the excerpts whenever they and the summary "
             "disagree on a detail. Answer ONLY using this supplied material. "
             "If the answer cannot be found in it, reply exactly: \"" + NOT_FOUND_ANSWER + "\" "
             "Do not use outside knowledge. Be concise and mention specific details (names, numbers) when they are available.")

_DEPENDENT_RE = re.compile(r"\b(it|its|they|them|their|this|that|these|those|he|she|such|there|former|latter|same)\b", re.I)
_NOT_FOUND_RE = re.compile(r"could not be found in the uploaded document|not (?:available|found|mentioned|stated|provided|specified) "
                           r"in the (?:uploaded )?(?:document|context|paper|text)", re.I)


class QuestionAnsweringService:
    def __init__(self, cfg, db, retriever, llm, extractive):
        self.cfg, self.db, self.retriever, self.llm, self.extractive = cfg, db, retriever, llm, extractive
        self.vstore = retriever.vstore

    def _summary_context(self, document_id):
        """The paper's summary, given to the LLM alongside the retrieved excerpts so it has the whole
        paper's shape (not just the top-K chunks) to reason with. Uses whichever summary is already
        cached (no extra LLM call); if none has been generated yet, falls back to a fast, no-LLM
        extractive summary (pure embedding similarity) so this never slows down or costs an extra call."""
        for kind in ("Brief", "Detailed", "Academic Summary"):
            cached = self.db.get_summary(document_id, kind)
            if cached:
                return cached
        try:
            chunks = [c for c in self.vstore.get_chunks(document_id) if c.get("section") not in SKIP_SECTIONS]
            sents = self.extractive.centroid_summary(chunks, 4)
            return " ".join(s["text"] for s in sents)
        except Exception as e:
            logger.warning("Could not build a fallback summary for QA context: %s", e)
            return ""

    def _needs_rewrite(self, question):
        return bool(_DEPENDENT_RE.search(question)) or len(question.split()) < 5

    def _standalone_question(self, question, history):
        """Resolve pronouns ('it', 'they'...) using only the last exchange."""
        if not history or not self._needs_rewrite(question):
            return question
        last_q = next((m["content"] for m in reversed(history) if m["role"] == "user"), "")
        last_a = next((m["content"] for m in reversed(history) if m["role"] == "assistant"), "")
        if not self.llm.generative:
            return f"{last_q} {question}".strip()
        try:
            rewritten = self.llm.generate(
                "Rewrite the user's latest question so it is fully self-contained, replacing pronouns and vague references with what "
                "they refer to in the previous exchange. Do NOT answer it. If it is already self-contained, return it unchanged. "
                "Output only the question.",
                f"Previous question: {last_q}\nPrevious answer: {last_a[:400]}\nLatest question: {question}",
                max_tokens=80, temperature=0.0)
            return rewritten.strip().strip('"') or question
        except AppError:
            return question

    @staticmethod
    def _sources(results):
        return [{"page": page_label(r), "page_start": r["page_number"], "section": r["section"],
                 "score": round(r["score"], 3), "text": r["text"][:450]} for r in results]

    def ask(self, document_id, question, top_k=None, threshold=None):
        start = time.time()
        question = (question or "").strip()
        if not question:
            raise AppError("Please type a question.")
        if len(question) > 1000:
            raise AppError("The question is too long. Please shorten it.")
        history = self.db.get_chat(document_id, limit=4)
        standalone = self._standalone_question(question, history)
        ret = self.retriever.retrieve(document_id, standalone, top_k, threshold)
        results = ret["results"]

        if not ret["relevant"]:                                   # out-of-context question: do NOT use general knowledge
            answer, answered, sources = NOT_FOUND_ANSWER, False, []
        else:
            if self.llm.generative:
                context = "\n\n".join(f"[Source {i} | Page {page_label(r)} | {r['section']}]\n{r['text']}"
                                      for i, r in enumerate(results, 1))
                summary_ctx = self._summary_context(document_id)
                prompt = ((f"[Paper summary]\n{summary_ctx}\n\n" if summary_ctx else "") +
                         f"[Relevant excerpts]\n{context}\n\nQuestion: {standalone}\n\nAnswer:")
                answer = self.llm.generate(QA_SYSTEM, prompt, max_tokens=500, temperature=0.0)
            else:
                ranked = self.extractive.rank(standalone, results[:3], n=3)
                answer = " ".join(s["text"] for s in sorted(ranked, key=lambda s: s["order"])) or NOT_FOUND_ANSWER
            answered = not (_NOT_FOUND_RE.search(answer) and len(answer) < 250)
            if not answered:
                answer = NOT_FOUND_ANSWER
            sources = self._sources(results) if answered else []

        self.db.add_chat(document_id, "user", question)
        self.db.add_chat(document_id, "assistant", answer, sources)
        return {"question": question, "standalone_question": standalone, "answer": answer, "answered": answered,
                "sources": sources, "top_score": ret["top_score"], "n_retrieved": len(results),
                "seconds": time.time() - start, "mode": "llm" if self.llm.generative else "extractive"}


retriever = RetrievalService(cfg, vstore)
summarizer = SummarizationService(cfg, db, vstore, llm, extractive)
keyinfo = KeyInformationService(cfg, db, vstore, llm, extractive, retriever)
qa = QuestionAnsweringService(cfg, db, retriever, llm, extractive)
print("Services ready.")

## 8. Evaluation
* **Summaries:** ROUGE-1/2/L and BERTScore (DistilBERT, lighter than the default) against a reference summary you provide.
* **Retrieval:** for each question the expected source pages are compared with the pages actually retrieved → **Precision, Recall, Hit@K**,
  plus response time, top similarity and number of retrieved chunks.
* The built-in demo questions use *silver labels* (pages where keywords occur most). For your report, replace them with manually verified pages
  using the format `question | 4,5`.

In [ ]:
from matplotlib.figure import Figure

DEMO_EVAL_QUESTIONS = [
    ("What dataset was used?", r"dataset|data set|corpus|articles"),
    ("What methodology was used?", r"methodolog|pipeline|tf-idf|vector"),
    ("What algorithms were used?", r"logistic regression|naive bayes|svm|support vector|random forest|neural|transformer"),
    ("What were the results?", r"accuracy|f1|precision|recall|results"),
    ("What are the limitations?", r"limitation|future work|generali[sz]e"),
    ("What is the conclusion?", r"conclusion|concluded|in this (?:paper|study|work)|this paper"),
]


class EvaluationService:
    def __init__(self, cfg, db, retriever, summarizer):
        self.cfg, self.db, self.retriever, self.summarizer = cfg, db, retriever, summarizer

    def summary_scores(self, reference: str, generated: str):
        if not reference.strip() or not generated.strip():
            raise AppError("Both a reference summary and a generated summary are required.")
        try:
            from rouge_score import rouge_scorer
        except ImportError:
            raise AppError("The 'rouge-score' package is not installed. Run the install cell.")
        scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
        rouge = {k: {"precision": v.precision, "recall": v.recall, "f1": v.fmeasure}
                 for k, v in scorer.score(reference, generated).items()}
        bert, note = None, ""
        try:
            from bert_score import score as bert_score_fn
            P, R, F = bert_score_fn([generated], [reference], model_type="distilbert-base-uncased", lang="en", verbose=False)
            bert = {"precision": float(P[0]), "recall": float(R[0]), "f1": float(F[0])}
        except Exception as e:
            note = "BERTScore is unavailable (package or model download failed)."
            logger.warning("BERTScore failed: %s", e)
        return {"rouge": rouge, "bertscore": bert, "note": note}

    def evaluate_summary(self, document_id, kind, reference):
        result = self.summarizer.summarize(document_id, kind, force=True)
        scores = self.summary_scores(reference, result["text"])
        scores.update({"generated": result["text"], "seconds": result["seconds"], "kind": kind})
        return scores

    @staticmethod
    def score_figure(scores):
        fig = Figure(figsize=(6, 3.2))
        ax = fig.subplots()
        labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L"] + (["BERTScore"] if scores.get("bertscore") else [])
        vals = [scores["rouge"][k]["f1"] for k in ("rouge1", "rouge2", "rougeL")] + ([scores["bertscore"]["f1"]] if scores.get("bertscore") else [])
        bars = ax.bar(labels, vals, color="#1F6F78")
        ax.set_ylim(0, 1); ax.set_ylabel("F1 score"); ax.set_title("Summary quality")
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
        fig.tight_layout()
        return fig

    # ---- retrieval evaluation ----
    def _silver_pages(self, document_id, pattern, n=2):
        counts = [(len(re.findall(pattern, p["text"], re.I)), p["page_number"]) for p in self.db.get_pages(document_id)]
        return sorted(pg for c, pg in sorted(counts, reverse=True)[:n] if c > 0)

    def default_dataset(self, document_id):
        return [{"question": q, "expected_pages": self._silver_pages(document_id, pat)} for q, pat in DEMO_EVAL_QUESTIONS]

    @staticmethod
    def parse_dataset(text):
        items = []
        for line in text.strip().splitlines():
            if "|" not in line:
                continue
            q, pages = line.split("|", 1)
            nums = sorted({int(x) for x in re.findall(r"\d+", pages)})
            if q.strip() and nums:
                items.append({"question": q.strip(), "expected_pages": nums})
        return items

    def evaluate_retrieval(self, document_id, dataset, k=3):
        rows = []
        for item in dataset:
            ret = self.retriever.retrieve(document_id, item["question"], top_k=k, threshold=0.0)
            covered = sorted({p for r in ret["results"] for p in range(r["page_number"], r["page_end"] + 1)})
            expected = set(item["expected_pages"])
            hit = int(bool(expected & set(covered)))
            inter = expected & set(covered)
            rows.append({"question": item["question"], "expected": sorted(expected), "retrieved": covered,
                         "hit": hit, "precision": len(inter) / len(covered) if covered else 0.0,
                         "recall": len(inter) / len(expected) if expected else 0.0,
                         "top_score": ret["top_score"], "n_chunks": len(ret["results"]), "seconds": ret["seconds"]})
        n = max(1, len(rows))
        summary = {f"Hit@{k}": sum(r["hit"] for r in rows) / n,
                   "Precision": sum(r["precision"] for r in rows) / n,
                   "Recall": sum(r["recall"] for r in rows) / n,
                   "Avg top similarity": sum(r["top_score"] for r in rows) / n,
                   "Avg retrieval time (s)": sum(r["seconds"] for r in rows) / n,
                   "Avg chunks retrieved": sum(r["n_chunks"] for r in rows) / n}
        return {"rows": rows, "summary": summary, "k": k}

    @staticmethod
    def retrieval_figure(result):
        fig = Figure(figsize=(6, 3.2))
        ax = fig.subplots()
        k = result["k"]
        keys = [f"Hit@{k}", "Precision", "Recall"]
        vals = [result["summary"][x] for x in keys]
        bars = ax.bar(keys, vals, color=["#1F6F78", "#7A9E9F", "#B08D57"])
        ax.set_ylim(0, 1.1); ax.set_title("Retrieval quality")
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
        fig.tight_layout()
        return fig


evaluator = EvaluationService(cfg, db, retriever, summarizer)
print("Evaluation service ready.")

## 9. Pipeline, sample paper and self-tests

In [ ]:
class DocumentPipeline:
    """upload → validate → store → extract → clean → chunk → embed → vector store."""

    def __init__(self, cfg, db, processor, chunker, vstore):
        self.cfg, self.db, self.processor, self.chunker, self.vstore = cfg, db, processor, chunker, vstore

    def process(self, path, filename):
        """Generator yielding (document_id, status message) so the UI can show progress."""
        self.processor.validate_file(path, filename)
        document_id = uuid.uuid4().hex
        stored = self.processor.save_upload(path, document_id)
        self.db.create_document(document_id, filename)
        yield document_id, "Document uploaded successfully."
        yield document_id, "Processing document..."
        try:
            self.db.update_status(document_id, "processing")
            pages = self.processor.extract_pages(stored, document_id)
            self.db.save_pages(document_id, pages)
            chunks = self.chunker.chunk_pages(document_id, pages)
            if not chunks:
                raise AppError("No readable text could be found in this PDF.")
            self.vstore.add_documents(chunks)
            self.vstore.save()
            self.db.update_counts(document_id, len(pages), sum(len(p["text"].split()) for p in pages))
            self.db.update_status(document_id, "ready")
        except Exception as e:
            msg = user_message(e)
            self.vstore.delete_document(document_id)
            self.db.update_status(document_id, "failed", msg)
            raise (e if isinstance(e, AppError) else AppError(msg))
        yield document_id, "Document ready."

    def process_file(self, path, filename=None):
        """Non-generator convenience wrapper. Returns the document_id."""
        doc_id = None
        for doc_id, _ in self.process(path, filename or Path(path).name):
            pass
        return doc_id

    def delete(self, document_id):
        self.vstore.delete_document(document_id)
        self.vstore.save()
        (self.cfg.upload_dir / f"{document_id}.pdf").unlink(missing_ok=True)
        self.db.delete_document(document_id)

    def overview(self, document_id):
        doc = self.db.get_document(document_id)
        if not doc:
            raise AppError("Document not found.")
        pages = self.db.get_pages(document_id)
        doc["empty_pages"] = [p["page_number"] for p in pages if not p["has_text"]]
        doc["chunks"] = len(self.vstore.get_chunks(document_id))
        return doc


pipeline = DocumentPipeline(cfg, db, processor, chunker, vstore)

# ---------- A built-in sample paper so the demo works without uploading anything ----------
SAMPLE_PAGES = [
"""Text Classification of News Articles Using TF-IDF and Machine Learning Models

Anita Sharma, Rahul Verma and Priya Nair
Department of Computer Applications, Example University

Keywords: text classification; TF-IDF; news articles; machine learning; logistic regression

Abstract
Automatic categorisation of news articles helps publishers organise large volumes of content. This paper studies whether classical machine learning models combined with TF-IDF features can classify news articles accurately. We compare Logistic Regression, Multinomial Naive Bayes and a Linear Support Vector Machine on the BBC News dataset. The Linear SVM achieved the best accuracy of 97.3 percent. The results show that simple, well-tuned models remain competitive for topic classification.

1. Introduction
News organisations publish thousands of articles every day and manually assigning each article to a category is slow and error prone. The research problem addressed in this work is how to categorise news articles automatically with high accuracy while keeping the models simple and fast to train. The objective of this study is to compare three classical classifiers on a public news corpus and to identify the most reliable one. The main contributions are a reproducible preprocessing pipeline and a detailed comparison of the three models.""",
"""2. Related Work
Earlier studies on text categorisation used bag-of-words features with Naive Bayes classifiers. Later work introduced Support Vector Machines, which handle high dimensional sparse data well. More recent research applies deep neural networks and transformer models, but these require far more computation and labelled data than classical methods.

3. Dataset
The experiments use the BBC News dataset, which contains 2,225 news articles published by the BBC between 2004 and 2005. Each article belongs to exactly one of five categories: business, entertainment, politics, sport and tech. The class distribution is fairly balanced, with sport having 511 articles and entertainment having 386 articles. The dataset is in English and was downloaded from a public repository.""",
"""4. Methodology
The proposed pipeline has four stages. First, the text is converted to lower case and punctuation and English stop words are removed. Second, the cleaned text is converted into numerical vectors using TF-IDF with unigrams and bigrams. Third, three classification algorithms are trained: Logistic Regression, Multinomial Naive Bayes and a Linear Support Vector Machine. Finally, the trained models are evaluated on unseen articles.

5. Experimental Setup
The dataset was split into 80 percent training and 20 percent testing data using stratified sampling with a fixed random seed of 42. The vocabulary was limited to the 10,000 most frequent terms. Hyperparameters were tuned with five-fold cross-validation on the training set. Performance was measured using accuracy, precision, recall and F1-score. All experiments were run in Python with the scikit-learn library on a laptop with 8 GB of memory.""",
"""6. Results
The Linear SVM achieved the highest accuracy of 97.3 percent, followed by Logistic Regression with 96.9 percent and Multinomial Naive Bayes with 95.7 percent. The macro F1-score of the Linear SVM was 0.97. Most errors occurred between the business and politics categories, which share vocabulary about economic policy.

7. Limitations
The study uses a single English dataset of modest size, so the findings may not generalise to other languages or larger corpora. Deep learning models were not evaluated, and the articles were assigned to only one category each.

8. Conclusion
This paper showed that TF-IDF features combined with a Linear SVM classify news articles with 97.3 percent accuracy. Future work will evaluate transformer based models and multilingual datasets.

References
[1] D. Greene and P. Cunningham. Practical solutions to the problem of diagonal dominance in kernel document clustering. ICML 2006.""",
]


def create_sample_paper(path=None) -> Path:
    import fitz
    path = Path(path or Path(cfg.data_dir) / "sample_paper.pdf")
    doc = fitz.open()
    for text in SAMPLE_PAGES:
        page = doc.new_page()
        for size in (10, 9, 8, 7):                        # shrink until the text fits the page
            if page.insert_textbox(fitz.Rect(50, 50, 545, 790), text, fontsize=size) >= 0:
                break
    doc.save(str(path))
    doc.close()
    return path


# ---------- Self-tests ----------
def run_self_tests(verbose=True):
    """Runs the test cases from the project specification against a temporary copy of the sample paper."""
    results, state = [], {}

    def check(name, fn):
        try:
            detail = fn()
            results.append((name, "PASS", detail or ""))
        except Exception as e:
            results.append((name, "FAIL", f"{type(e).__name__}: {e}"))

    def t_extraction():
        state["pdf"] = create_sample_paper(Path(cfg.data_dir) / "selftest_paper.pdf")
        pages = processor.extract_pages(state["pdf"], "test")
        assert len(pages) == 4, f"expected 4 pages, got {len(pages)}"
        assert pages[0]["page_number"] == 1 and "Text Classification" in pages[0]["text"]
        assert all(p["has_text"] for p in pages)
        return "4 pages, page numbers preserved"

    def t_validation():
        bad = Path(cfg.data_dir) / "bad.txt"; bad.write_text("hello")
        empty = Path(cfg.data_dir) / "empty.pdf"; empty.write_bytes(b"")
        corrupt = Path(cfg.data_dir) / "corrupt.pdf"; corrupt.write_bytes(b"this is not a pdf at all")
        for p in (bad, empty, corrupt):
            try:
                processor.validate_file(p)
            except AppError:
                continue
            raise AssertionError(f"{p.name} should have been rejected")
        for p in (bad, empty, corrupt):
            p.unlink(missing_ok=True)
        return "wrong type, empty and corrupted files rejected"

    def t_cleaning():
        raw = "Multi   spaces\there\n\n\n\n12\nhyphen-\nated word  \u00a0 end \ufb01nal"
        out = cleaner.clean(raw)
        assert "  " not in out and "\n\n\n" not in out
        assert "hyphenated" in out and "final" in out
        assert "\n12\n" not in f"\n{out}\n", "bare page number should be removed"
        return "spaces, blank lines, hyphenation, page numbers, ligatures handled"

    def t_chunking():
        pages = processor.extract_pages(state["pdf"], "test")
        small = ChunkingService(Config(chunk_size=150, chunk_overlap=30)).chunk_pages("test1234", pages)
        assert len(small) > 3
        assert all(c["chunk_id"] and c["text"] and c["page_number"] >= 1 for c in small)
        assert max(len(c["text"].split()) for c in small) <= int(150 / 1.3) + 1
        assert any(c["section"] == "Dataset" for c in small), "section detection failed"
        return f"{len(small)} chunks (size limit and sections OK)"

    def t_embedding():
        e = embedder.encode(["hello world", "another sentence"])
        assert e.shape[0] == 2 and abs(np.linalg.norm(e[0]) - 1) < 1e-4
        return f"backend={embedder.backend}, dim={e.shape[1]}"

    def t_pipeline_and_search():
        state["doc"] = pipeline.process_file(state["pdf"], "selftest_paper.pdf")
        ov = pipeline.overview(state["doc"])
        assert ov["processing_status"] == "ready" and ov["page_count"] == 4 and ov["chunks"] >= 2
        res = vstore.search("What dataset was used?", 3, state["doc"])
        assert res and all("score" in r and "page_number" in r and "document_id" in r for r in res)
        assert any(r["page_number"] <= 2 <= r["page_end"] for r in res), "dataset page (2) was not retrieved"
        return f"{ov['chunks']} chunks stored; dataset question retrieved page(s) {[page_label(r) for r in res]}"

    def t_store_persistence():
        vstore.save()
        fresh = VectorStore(embedder, cfg.vector_dir)
        assert fresh.load() and len(fresh.chunks) == len(vstore.chunks)
        assert fresh.search("dataset", 1, state["doc"])
        return "save() / load() round trip OK"

    def t_summaries():
        for kind in SummarizationService.TYPES:
            r = summarizer.summarize(state["doc"], kind, force=True)
            assert len(r["text"].split()) >= 8, f"{kind} summary too short"
        return "brief, detailed, bullet and academic summaries generated"

    def t_key_info():
        data = keyinfo.extract(state["doc"], force=True)
        assert set(data) == set(INFO_FIELDS)
        assert data["title"] != NOT_FOUND, "title missing"
        return f"all {len(INFO_FIELDS)} fields present; title = {str(data['title'])[:60]!r}"

    def t_qa():
        questions = ["What is the research problem?", "What dataset was used?", "What methodology was used?",
                     "What algorithms were used?", "What were the results?", "What are the limitations?", "What is the conclusion?"]
        pages_seen = []
        for q in questions:
            r = qa.ask(state["doc"], q)
            assert r["answered"] and r["answer"].strip(), f"no answer for: {q}"
            assert r["sources"], f"no sources for: {q}"
            pages_seen.append(r["sources"][0]["page"])
        return f"7 questions answered with sources (first-source pages: {pages_seen})"

    def t_followup():
        db.clear_chat(state["doc"])
        qa.ask(state["doc"], "What dataset was used?")
        r = qa.ask(state["doc"], "How many categories does it contain?")
        assert r["standalone_question"] != r["question"], "follow-up was not rewritten"
        return f"follow-up rewritten as: {r['standalone_question']!r}"

    def t_out_of_context():
        r = qa.ask(state["doc"], "Who is the Prime Minister of India?")
        assert not r["answered"] and r["answer"] == NOT_FOUND_ANSWER and r["sources"] == []
        return "unrelated question refused"

    def t_evaluation():
        ds = evaluator.default_dataset(state["doc"])
        res = evaluator.evaluate_retrieval(state["doc"], ds, k=3)
        assert 0 <= res["summary"]["Hit@3"] <= 1
        sc = evaluator.summary_scores("The Linear SVM reached 97.3 percent accuracy on BBC news.", "Linear SVM had 97.3 percent accuracy on the BBC News dataset.")
        assert sc["rouge"]["rouge1"]["f1"] > 0.3
        return f"Hit@3={res['summary']['Hit@3']:.2f}, ROUGE-1 F1={sc['rouge']['rouge1']['f1']:.2f}"

    def t_delete():
        pipeline.delete(state["doc"])
        assert not vstore.has_document(state["doc"]) and db.get_document(state["doc"]) is None
        return "document, vectors and file removed"

    for name, fn in [("PDF extraction", t_extraction), ("Upload validation", t_validation), ("Text cleaning", t_cleaning),
                     ("Chunking", t_chunking), ("Embedding generation", t_embedding), ("Pipeline + vector search", t_pipeline_and_search),
                     ("Vector store save/load", t_store_persistence), ("Summarisation (4 types)", t_summaries),
                     ("Key information extraction", t_key_info), ("Question answering (7 questions)", t_qa),
                     ("Follow-up question", t_followup), ("Out-of-context question", t_out_of_context),
                     ("Evaluation metrics", t_evaluation), ("Delete document", t_delete)]:
        check(name, fn)
        if name == "PDF extraction" and results[-1][1] == "FAIL":
            break                                       # later tests need the sample PDF
    if state.get("doc") and db.get_document(state["doc"]):
        pipeline.delete(state["doc"])                   # clean up if a test stopped early
    Path(cfg.data_dir, "selftest_paper.pdf").unlink(missing_ok=True)

    if verbose:
        print(f"\n{'Test':<36}{'Result':<8}Details\n" + "-" * 100)
        for n, s, d in results:
            print(f"{n:<36}{s:<8}{d}")
        passed = sum(1 for r in results if r[1] == "PASS")
        print("-" * 100 + f"\n{passed}/{len(results)} tests passed   (LLM: {llm.name}, embeddings: {embedder.backend})")
    return results

In [ ]:
# Set RUN_SELF_TESTS=0 in your environment to skip. Tests use a temporary copy of the sample paper and clean up after themselves.
RUN_SELF_TESTS = os.getenv("RUN_SELF_TESTS", "1") == "1"
if RUN_SELF_TESTS:
    _ = run_self_tests()
else:
    print("Self-tests skipped. Call run_self_tests() to run them.")

## 10. Web interface
Run the next cell to open the dashboard (inline in Jupyter/Colab, or at the printed local URL).
Tabs follow the specification: **Home · Upload Document · Summary · Key Information · Ask Document · Evaluation · History**.

This cell also defines a small stylesheet and HTML helper functions (teal masthead, status chips, step cards, a key-information
grid, source citation cards) that give every tab a consistent look, plus a real chat log (`gr.Chatbot`) for Ask Document instead
of a scrolling block of markdown.

In [ ]:
import html as _html
import gradio as gr

ACTIVE = {"id": None}         # the document currently selected (single-user demo app)
SHARE_PUBLIC_LINK = False     # set True to create a temporary public Gradio link (handy in Colab)

PRIMARY = "#1F6F78"           # teal -- matches the Phase-1 frontend palette, carried through the whole dashboard
PRIMARY_DARK = "#154E54"
INK = "#1B2A3A"
PAPER = "#F7F8FA"
RULE = "#E2E7EB"

# ------------------------------------------------------------------------------------------------
# Visual design
#
# A single stylesheet + small HTML helpers give the dashboard a consistent look without needing a
# separate frontend: a dark teal masthead with the project title, a status strip (model, document,
# page/word counts) under every tab, labelled step cards for the upload pipeline, a two-column
# "excerpt card" layout for cited sources, and a real chat bubble log (gr.Chatbot) for Ask Document
# instead of a scrolling wall of markdown. Colors and type stay consistent with the Phase-1 README
# (serif headings, teal accent) so the notebook and the original frontend plan read as one project.
# ------------------------------------------------------------------------------------------------
CUSTOM_CSS = f"""
@import url('https://fonts.googleapis.com/css2?family=Source+Serif+4:wght@600;700&family=Inter:wght@400;500;600&display=swap');

/* ---- force a consistent light appearance ----
   Gradio auto-switches its own text-color variables to a light-on-dark palette when the viewer's
   OS/browser is set to dark mode. This dashboard's cards always have a light (white) background, so
   that auto-switch would otherwise turn ordinary text (gr.Markdown output, labels, table text) white
   on a white card -- invisible until a hover/focus state happens to apply a different color. Setting
   color-scheme and every text-color variable explicitly keeps the page readable regardless of the
   viewer's system theme. */
:root, .dark, .gradio-container {{
    color-scheme: light !important;
    --body-text-color: {INK} !important;
    --body-text-color-subdued: #5b6b76 !important;
    --block-label-text-color: {INK} !important;
    --block-title-text-color: {INK} !important;
    --block-info-text-color: #5b6b76 !important;
    --input-text-color: {INK} !important;
    --body-background-fill: {PAPER} !important;
    --background-fill-primary: #fff !important;
    --background-fill-secondary: {PAPER} !important;
    --border-color-primary: {RULE} !important;
    --link-text-color: {PRIMARY_DARK} !important;
}}
.gradio-container {{ font-family: 'Inter', ui-sans-serif, system-ui, sans-serif !important; background: {PAPER} !important; color: {INK} !important; }}

/* gr.Markdown output (the Summary tab's rendered text, etc.) renders inside ".prose" -- force dark,
   readable text there too, regardless of the color Gradio's dark-mode detection would otherwise pick. */
.gradio-container .prose,
.gradio-container .prose p,
.gradio-container .prose li,
.gradio-container .prose strong,
.gradio-container .prose em,
.gradio-container .prose h1, .gradio-container .prose h2, .gradio-container .prose h3, .gradio-container .prose h4,
.gradio-container .prose blockquote,
.gradio-container .prose table, .gradio-container .prose td, .gradio-container .prose th {{
    color: {INK} !important;
}}
.gradio-container .prose sub {{ color: #5b6b76 !important; }}
/* Labels, checkbox/radio text, plain text and the file-upload dropzone all sit on this theme's light
   background, so their text must stay dark regardless of the viewer's OS/browser dark-mode setting.
   Anything inside a <button> is deliberately excluded (":not(button *)") and left to Gradio's own
   button theming, which already gives primary/stop buttons light text on their colored background. */
.gradio-container label:not(button *),
.gradio-container span:not(button *),
.gradio-container p:not(button *),
.gradio-container div:not(button *),
.gradio-container li:not(button *) {{ color: {INK} !important; }}

/* Typed/typable text (the question box, textareas, dropdowns) and its placeholder: dark text on the
   light input background, regardless of dark mode. -webkit-text-fill-color covers Chrome/Safari,
   which otherwise keep their own autofill/dark-mode text color even when "color" is overridden. */
.gradio-container input,
.gradio-container textarea,
.gradio-container select {{ color: {INK} !important; background: #fff !important; -webkit-text-fill-color: {INK} !important; }}
.gradio-container input::placeholder,
.gradio-container textarea::placeholder {{ color: #64748b !important; opacity: 1 !important; -webkit-text-fill-color: #64748b !important; }}

/* The file-upload dropzone ("Drop File Here / Click to Upload") and its icon: same light background,
   same forced-dark text, covered again here in case its markup uses a tag not listed above. */
.gradio-container .wrap, .gradio-container .wrap *,
.gradio-container .upload-container, .gradio-container .upload-container * {{ color: {INK} !important; }}

/* ---- masthead ---- */
.rpa-hero {{
    background: linear-gradient(135deg, {PRIMARY_DARK} 0%, {PRIMARY} 100%) !important;
    color: #F2F7F7 !important; border-radius: 18px; padding: 28px 32px; margin-bottom: 4px;
}}
.rpa-hero h1 {{
    font-family: 'Source Serif 4', Georgia, serif; font-weight: 700; font-size: 1.9rem;
    margin: 0 0 6px 0; letter-spacing: -0.01em; color: #F2F7F7 !important;
}}
.rpa-hero p {{ margin: 0; color: #CFE6E6 !important; font-size: 0.95rem; max-width: 56ch; }}

/* ---- status strip (shown under the masthead on every tab) ---- */
.rpa-status {{ display: flex; flex-wrap: wrap; gap: 10px; margin: 14px 0 6px 0; }}
.rpa-chip {{
    background: #fff !important; border: 1px solid {RULE}; border-radius: 10px; padding: 8px 14px;
    font-size: 0.82rem; color: {INK} !important; display: flex; align-items: baseline; gap: 6px; line-height: 1.3;
}}
.rpa-chip b {{ font-size: 0.95rem; color: {INK} !important; }}
.rpa-chip.rpa-chip-empty {{ color: #6b7a86 !important; font-style: italic; }}
.rpa-pill {{ border-radius: 999px; padding: 2px 10px; font-size: 0.78rem; font-weight: 500; }}
.rpa-pill-llm {{ background: #E3F1F2 !important; color: {PRIMARY_DARK} !important; }}
.rpa-pill-extractive {{ background: #FBEEE0 !important; color: #8A5A22 !important; }}
.rpa-pill-ready {{ background: #E6F4E6 !important; color: #2E6B34 !important; }}
.rpa-pill-warn {{ background: #FDEBEC !important; color: #A33A3F !important; }}

/* ---- section headings inside a tab ---- */
.rpa-section-title {{ font-family: 'Source Serif 4', Georgia, serif; font-size: 1.25rem; color: {INK} !important; margin: 6px 0 2px 0; }}
.rpa-section-sub {{ color: #475569 !important; font-size: 0.88rem; margin: 0 0 10px 0; }}

/* ---- upload pipeline stepper ---- */
.rpa-steps {{ display: flex; gap: 8px; margin: 10px 0 4px 0; }}
.rpa-step {{
    flex: 1; border-radius: 10px; padding: 10px 12px; border: 1px solid {RULE}; background: #fff !important;
    font-size: 0.82rem; color: #64748b !important; position: relative;
}}
.rpa-step .n {{ display:inline-block; width:20px; height:20px; border-radius:50%; background:#EDF1F3; color:#64748b !important;
               text-align:center; line-height:20px; font-size:0.74rem; margin-right:6px; }}
.rpa-step.done {{ border-color: {PRIMARY}; background: #F1F8F8 !important; color: {INK} !important; }}
.rpa-step.done .n {{ background: {PRIMARY} !important; color: #fff !important; }}
.rpa-step.active {{ border-color: {PRIMARY}; box-shadow: 0 0 0 2px #CFE6E6; color: {INK} !important; font-weight: 500; }}
.rpa-step.active .n {{ background: {PRIMARY_DARK} !important; color: #fff !important; }}
.rpa-step.failed {{ border-color: #D96B6B; background: #FDF1F1 !important; color: #A33A3F !important; }}
.rpa-step.failed .n {{ background: #D96B6B !important; color: #fff !important; }}

/* ---- document info cards ---- */
.rpa-stat-row {{ display: flex; gap: 12px; flex-wrap: wrap; margin: 4px 0 14px 0; }}
.rpa-stat {{ flex: 1; min-width: 120px; background: #fff !important; border: 1px solid {RULE}; border-radius: 12px; padding: 12px 16px; }}
.rpa-stat .v {{ font-size: 1.35rem; font-weight: 600; color: {INK} !important; }}
.rpa-stat .l {{ font-size: 0.78rem; color: #475569 !important; margin-top: 2px; }}

/* ---- key-information grid ---- */
.rpa-info-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap: 12px; margin-top: 6px; }}
.rpa-info-card {{ background: #fff !important; border: 1px solid {RULE}; border-left: 3px solid {PRIMARY}; border-radius: 10px; padding: 12px 16px; }}
.rpa-info-card.empty {{ border-left-color: {RULE}; color: #94a3b8 !important; }}
.rpa-info-card .f {{ font-size: 0.78rem; color: #475569 !important; margin-bottom: 4px; }}
.rpa-info-card .val {{ font-size: 0.92rem; color: {INK} !important; line-height: 1.45; }}
.rpa-info-card.empty .val {{ color: #94a3b8 !important; }}
.rpa-info-card .val ul {{ margin: 2px 0 0 0; padding-left: 18px; }}
.rpa-info-wide {{ grid-column: 1 / -1; }}

/* ---- source / citation cards ---- */
.rpa-source-card {{ background: #fff !important; border: 1px solid {RULE}; border-radius: 10px; padding: 12px 16px; margin-bottom: 10px; }}
.rpa-source-head {{ display: flex; justify-content: space-between; font-size: 0.8rem; color: {PRIMARY_DARK} !important; font-weight: 500; margin-bottom: 6px; }}
.rpa-source-text {{ font-size: 0.88rem; color: #334155 !important; line-height: 1.5; font-style: italic; }}
.rpa-empty-note {{ color: #475569 !important; font-style: italic; font-size: 0.9rem; }}

/* ---- evaluation tables ---- */
.rpa-table table {{ width: 100%; border-collapse: collapse; font-size: 0.86rem; }}
.rpa-table th, .rpa-table td {{ border-bottom: 1px solid {RULE}; padding: 6px 10px; text-align: left; color: {INK} !important; }}
.rpa-table th {{ color: #475569 !important; font-weight: 500; }}
.rpa-summary-line {{ margin-top: 8px; font-size: 0.88rem; color: {INK} !important; }}

/* ---- home page feature list ---- */
.rpa-flow {{ display: flex; align-items: stretch; gap: 6px; flex-wrap: wrap; margin: 10px 0 18px 0; }}
.rpa-flow .node {{ background: #fff !important; border: 1px solid {RULE}; border-radius: 10px; padding: 10px 14px; font-size: 0.82rem; color: {INK} !important; }}
.rpa-flow .arrow {{ align-self: center; color: #64748b !important; }}
.rpa-feature-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 14px; margin-top: 8px; }}
.rpa-feature {{ background: #fff !important; border: 1px solid {RULE}; border-radius: 12px; padding: 16px 18px; }}
.rpa-feature .t {{ font-weight: 600; color: {INK} !important; margin-bottom: 4px; }}
.rpa-feature .d {{ font-size: 0.86rem; color: #475569 !important; line-height: 1.45; }}
"""

UPLOAD_STEPS = ["Validate file", "Extract & clean text", "Chunk & embed", "Ready"]
_UPLOAD_STATUS_TO_STEP = {"Document uploaded successfully.": 1, "Processing document...": 2, "Document ready.": 3}


def _esc(x) -> str:
    return _html.escape(str(x)) if x is not None else ""


# ---------- shared HTML builders ----------
def render_steps(step_index: int, failed: bool = False) -> str:
    """step_index: 0 = nothing done yet .. len(UPLOAD_STEPS)-1 = last step active/failed."""
    cells = []
    for i, label in enumerate(UPLOAD_STEPS):
        cls = "rpa-step"
        if failed and i == step_index:
            cls += " failed"
        elif i < step_index:
            cls += " done"
        elif i == step_index:
            cls += " active" if not failed else ""
        cells.append(f'<div class="{cls}"><span class="n">{i + 1}</span>{_esc(label)}</div>')
    return f'<div class="rpa-steps">{"".join(cells)}</div>'


def render_status_strip():
    """The chip row shown under the masthead: active document + LLM mode, kept in sync on every tab."""
    mode_pill = (f'<span class="rpa-pill rpa-pill-llm">{_esc(llm.name)}</span>' if llm.generative else
                '<span class="rpa-pill rpa-pill-extractive">extractive (no LLM key set)</span>')
    chips = [f'<div class="rpa-chip">Model <b></b>{mode_pill}</div>']
    d = db.get_document(ACTIVE["id"]) if ACTIVE["id"] else None
    if not d:
        chips.append('<div class="rpa-chip rpa-chip-empty">No document selected -- upload one, or load the sample paper</div>')
    else:
        ready = d["processing_status"] == "ready"
        status_pill = (f'<span class="rpa-pill rpa-pill-ready">{_esc(d["processing_status"])}</span>' if ready else
                       f'<span class="rpa-pill rpa-pill-warn">{_esc(d["processing_status"])}</span>')
        chips.append(f'<div class="rpa-chip"><b>{_esc(d["filename"])}</b></div>')
        chips.append(f'<div class="rpa-chip">{d["page_count"]} pages &nbsp;&middot;&nbsp; {d["word_count"]:,} words</div>')
        chips.append(f'<div class="rpa-chip">{status_pill}</div>')
    return f'<div class="rpa-status">{"".join(chips)}</div>'


def render_stat_row(document_id):
    d = pipeline.overview(document_id)
    stats = [("Pages", d["page_count"]), ("Words", f'{d["word_count"]:,}'), ("Chunks indexed", d["chunks"]),
             ("Status", d["processing_status"].capitalize())]
    cards = "".join(f'<div class="rpa-stat"><div class="v">{_esc(v)}</div><div class="l">{_esc(l)}</div></div>' for l, v in stats)
    note = (f'<p class="rpa-section-sub">Pages with no extractable text: {_esc(d["empty_pages"])}</p>' if d["empty_pages"] else "")
    return f'<div class="rpa-stat-row">{cards}</div>{note}'


def fmt_value(v):
    if isinstance(v, list):
        return "\n".join(f"- {x}" for x in v)
    return str(v)


def _value_html(v):
    if isinstance(v, list):
        return "<ul>" + "".join(f"<li>{_esc(x)}</li>" for x in v) + "</ul>"
    return _esc(v).replace("\n", "<br>")


KEY_INFO_TITLES = {"title": "Title", "authors": "Authors", "abstract": "Abstract", "research_problem": "Research problem",
                   "objective": "Objective", "methodology": "Methodology", "dataset": "Dataset",
                   "algorithms_models": "Algorithms / models", "experimental_setup": "Experimental setup",
                   "results": "Results", "important_findings": "Important findings", "limitations": "Limitations",
                   "conclusion": "Conclusion", "keywords": "Keywords"}
_WIDE_FIELDS = {"abstract", "methodology", "results"}


def render_key_info(data) -> str:
    if not data:
        return '<p class="rpa-empty-note">No key information extracted yet.</p>'
    cards = []
    for f in INFO_FIELDS:
        v = data.get(f, NOT_FOUND)
        empty = v == NOT_FOUND
        wide = " rpa-info-wide" if f in _WIDE_FIELDS else ""
        cards.append(f'<div class="rpa-info-card{" empty" if empty else ""}{wide}">'
                     f'<div class="f">{_esc(KEY_INFO_TITLES[f])}</div><div class="val">{_value_html(v)}</div></div>')
    return f'<div class="rpa-info-grid">{"".join(cards)}</div>'


def render_sources(sources) -> str:
    if not sources:
        return '<p class="rpa-empty-note">No sources: the answer was not found in the document.</p>'
    cards = []
    for i, s in enumerate(sources, 1):
        cards.append(f'<div class="rpa-source-card"><div class="rpa-source-head">'
                     f'<span>Source {i} &middot; page {_esc(s["page"])} &middot; {_esc(s["section"])}</span>'
                     f'<span>similarity {s["score"]:.2f}</span></div>'
                     f'<div class="rpa-source-text">&ldquo;{_esc(s["text"])}&hellip;&rdquo;</div></div>')
    return "".join(cards)


def doc_choices():
    return [(f"{d['filename']}  ({d['upload_date'][:16].replace('T', ' ')})  [{d['processing_status']}]", d["document_id"])
            for d in db.list_documents()]


def fmt_doc_info_md(document_id):
    d = pipeline.overview(document_id)
    lines = [f"| | |\n|---|---|\n| **Filename** | {d['filename']} |\n| **Pages** | {d['page_count']} |\n| **Word count** | {d['word_count']:,} |",
             f"| **Status** | {d['processing_status']} |\n| **Chunks indexed** | {d['chunks']} |"]
    out = "\n".join(lines)
    if d["empty_pages"]:
        out += f"\n\nPages with no extractable text: {d['empty_pages']}"
    return out


def chat_history_for_chatbot(document_id):
    """Builds the list-of-dicts format gr.Chatbot(type='messages') expects, from the stored chat log."""
    msgs = db.get_chat(document_id) if document_id else []
    out = []
    for m in msgs:
        if m["role"] == "user":
            out.append({"role": "user", "content": m["content"]})
        else:
            pages = ", ".join(dict.fromkeys(s["page"] for s in m["sources"]))
            content = m["content"] + (f"\n\n*Sources: page {pages}*" if pages else "")
            out.append({"role": "assistant", "content": content})
    return out


def render_page(document_id, page_number):
    """Renders a PDF page as an image so a citation can be inspected."""
    import fitz
    from PIL import Image
    pdf = cfg.upload_dir / f"{document_id}.pdf"
    if not pdf.exists():
        raise AppError("The original PDF is no longer available.")
    doc = fitz.open(str(pdf))
    try:
        n = int(min(max(1, page_number), doc.page_count))
        pix = doc[n - 1].get_pixmap(dpi=110)
        return Image.open(io.BytesIO(pix.tobytes("png")))
    finally:
        doc.close()


def export_text(document_id, suffix, text):
    d = db.get_document(document_id)
    path = cfg.export_dir / f"{Path(d['filename']).stem}_{suffix}"
    path.write_text(text, encoding="utf-8")
    return str(path)


def require_active():
    if not ACTIVE["id"] or not db.get_document(ACTIVE["id"]):
        raise AppError("Please upload or select a document first (Upload Document or History tab).")
    doc = db.get_document(ACTIVE["id"])
    if doc["processing_status"] != "ready":
        raise AppError("This document is not ready. Please upload it again.")
    return ACTIVE["id"]


def warn(msg): return f'<p class="rpa-empty-note" style="color:#A33A3F;font-style:normal;">{_esc(msg)}</p>'


# ---------- callbacks ----------
def _run_upload(path_or_none, filename_override=None):
    """Shared by the file-upload and sample-paper buttons; yields (steps_html, stat_row_html, status_strip_html, history_dropdown)."""
    if path_or_none is None:
        yield render_steps(0), warn("Please choose a PDF file first."), render_status_strip(), gr.update()
        return
    path = path_or_none if isinstance(path_or_none, str) else getattr(path_or_none, "name", str(path_or_none))
    filename = filename_override or Path(path).name
    try:
        doc_id = None
        for doc_id, status in pipeline.process(path, filename):
            step = _UPLOAD_STATUS_TO_STEP.get(status, 0)
            ACTIVE["id"] = doc_id if status == "Document ready." else ACTIVE["id"]
            yield render_steps(step), "", render_status_strip(), gr.update()
        yield render_steps(3), render_stat_row(doc_id), render_status_strip(), gr.update(choices=doc_choices(), value=doc_id)
    except Exception as e:
        yield render_steps(0, failed=True), warn(user_message(e)), render_status_strip(), gr.update()


def cb_upload(file):
    yield from _run_upload(file)


def cb_sample():
    yield from _run_upload(str(create_sample_paper()), "sample_paper.pdf")


def cb_summary(kind, include_info):
    try:
        doc_id = require_active()
        result = summarizer.summarize(doc_id, kind)
        parts = [f"## {kind}", result["text"]]
        if include_info:
            info = keyinfo.extract(doc_id)
            parts.append("## Research information\n" + "\n\n".join(
                f"**{t}:** {fmt_value(info[f])}" for t, f in [("Objective", "objective"), ("Methodology", "methodology"),
                                                              ("Dataset", "dataset"), ("Results", "results"), ("Conclusion", "conclusion")]))
        parts.append(f"<sub>Generated in {result['mode']} mode, {result['seconds']:.1f}s.</sub>")
        text = "\n\n".join(parts)
        return text, export_text(doc_id, f"{kind.lower().replace(' ', '_')}_summary.md", text), render_status_strip()
    except Exception as e:
        return warn(user_message(e)).replace("<p", "<p ").replace('class="rpa-empty-note"', 'class="rpa-empty-note"'), None, render_status_strip()


def cb_keyinfo(force):
    try:
        doc_id = require_active()
        data = keyinfo.extract(doc_id, force=force)
        return render_key_info(data), data, export_text(doc_id, "key_information.json", json.dumps(data, indent=2)), render_status_strip()
    except Exception as e:
        return warn(user_message(e)), None, None, render_status_strip()


def cb_ask(question, history, top_k, threshold):
    history = history or []
    try:
        doc_id = require_active()
        r = qa.ask(doc_id, question, int(top_k), float(threshold))
        meta = (f"{r['seconds']:.1f}s &middot; top similarity {r['top_score']:.2f} &middot; {r['n_retrieved']} chunks retrieved &middot; "
                f"{r['mode']} mode" + (f" &middot; interpreted as: &ldquo;{_esc(r['standalone_question'])}&rdquo;"
                                       if r["standalone_question"] != r["question"] else ""))
        img = render_page(doc_id, r["sources"][0]["page_start"]) if r["sources"] else None
        history = history + [{"role": "user", "content": question}, {"role": "assistant", "content": r["answer"]}]
        src_html = render_sources(r["sources"]) + f'<p class="rpa-section-sub">{meta}</p>'
        return "", history, src_html, img, render_status_strip()
    except Exception as e:
        history = history + [{"role": "assistant", "content": user_message(e)}]
        return question, history, warn(user_message(e)), None, render_status_strip()


def cb_clear_chat():
    if ACTIVE["id"]:
        db.clear_chat(ACTIVE["id"])
    return [], "", "", None


def cb_view_page(page_number):
    try:
        return render_page(require_active(), page_number or 1)
    except Exception as e:
        gr.Warning(user_message(e))
        return None


def _table_html(headers, rows):
    head = "".join(f"<th>{_esc(h)}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td>{c}</td>" for c in row) + "</tr>" for row in rows)
    return f'<div class="rpa-table"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'


def cb_eval_summary(kind, reference):
    try:
        doc_id = require_active()
        res = evaluator.evaluate_summary(doc_id, kind, reference)
        rows = [[name, f'{res["rouge"][k]["precision"]:.3f}', f'{res["rouge"][k]["recall"]:.3f}', f'{res["rouge"][k]["f1"]:.3f}']
                for name, k in [("ROUGE-1", "rouge1"), ("ROUGE-2", "rouge2"), ("ROUGE-L", "rougeL")]]
        if res["bertscore"]:
            b = res["bertscore"]; rows.append(["BERTScore", f'{b["precision"]:.3f}', f'{b["recall"]:.3f}', f'{b["f1"]:.3f}'])
        md = (_table_html(["Metric", "Precision", "Recall", "F1"], rows) +
              f'<p class="rpa-summary-line">Generation time: {res["seconds"]:.1f}s {_esc(res["note"])}</p>')
        return res["generated"], reference, md, evaluator.score_figure(res)
    except Exception as e:
        return "", reference, warn(user_message(e)), None


def cb_eval_retrieval(dataset_text, k):
    try:
        doc_id = require_active()
        dataset = evaluator.parse_dataset(dataset_text) if dataset_text.strip() else evaluator.default_dataset(doc_id)
        dataset = [d for d in dataset if d["expected_pages"]]
        if not dataset:
            raise AppError("No questions with expected pages. Use the format: question | 4,5")
        res = evaluator.evaluate_retrieval(doc_id, dataset, int(k))
        rows = [[_esc(r["question"]), _esc(r["expected"]), _esc(r["retrieved"]), r["hit"], f'{r["precision"]:.2f}',
                f'{r["recall"]:.2f}', f'{r["top_score"]:.2f}', f'{r["seconds"] * 1000:.0f} ms'] for r in res["rows"]]
        md = (_table_html(["Question", "Expected pages", "Retrieved pages", f"Hit@{k}", "Precision", "Recall", "Top sim.", "Time"], rows) +
              '<p class="rpa-summary-line">' + " &middot; ".join(f"<b>{_esc(name)}:</b> {val:.2f}" for name, val in res["summary"].items()) + "</p>")
        return md, evaluator.retrieval_figure(res)
    except Exception as e:
        return warn(user_message(e)), None


def cb_default_dataset():
    try:
        ds = evaluator.default_dataset(require_active())
        return "\n".join(f"{d['question']} | {', '.join(map(str, d['expected_pages']))}" for d in ds if d["expected_pages"])
    except Exception as e:
        gr.Warning(user_message(e))
        return ""


def cb_history_load(document_id):
    if not document_id:
        return render_status_strip(), [], ""
    ACTIVE["id"] = document_id
    try:
        return render_status_strip(), chat_history_for_chatbot(document_id), render_stat_row(document_id)
    except Exception as e:
        return render_status_strip(), [], warn(user_message(e))


def cb_history_delete(document_id):
    try:
        if not document_id:
            raise AppError("Select a document first.")
        pipeline.delete(document_id)
        if ACTIVE["id"] == document_id:
            ACTIVE["id"] = None
        return render_status_strip(), gr.update(choices=doc_choices(), value=None), "Document deleted.", ""
    except Exception as e:
        return render_status_strip(), gr.update(), warn(user_message(e)), ""


def cb_refresh():
    return gr.update(choices=doc_choices())


# ---------- layout ----------
THEME = gr.themes.Soft(
    primary_hue=gr.themes.colors.teal, secondary_hue=gr.themes.colors.slate, neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace", "monospace"],
).set(button_primary_background_fill=PRIMARY, button_primary_background_fill_hover=PRIMARY_DARK,
      block_title_text_weight="600", block_label_text_size="*text_sm")


def build_ui():
    with gr.Blocks(title="Research Paper Assistant", theme=THEME, css=CUSTOM_CSS) as demo:
        gr.HTML('<div class="rpa-hero"><h1>Research Paper Assistant</h1>'
               '<p>Upload a research paper and get summaries, structured key information, and grounded '
               'question answering -- every answer is drawn only from the paper, with the page it came from.</p></div>')
        status_strip = gr.HTML(render_status_strip())

        with gr.Tabs():
            # ---------------- Home ----------------
            with gr.Tab("Home"):
                gr.HTML(f"""
                <p class="rpa-section-title">How it works</p>
                <div class="rpa-flow">
                    <div class="node">PDF upload</div><div class="arrow">&rarr;</div>
                    <div class="node">Text extraction &amp; cleaning</div><div class="arrow">&rarr;</div>
                    <div class="node">Chunking &amp; embeddings</div><div class="arrow">&rarr;</div>
                    <div class="node">Vector search (FAISS)</div><div class="arrow">&rarr;</div>
                    <div class="node">LLM</div><div class="arrow">&rarr;</div>
                    <div class="node">Answer + page citation</div>
                </div>
                <div class="rpa-feature-grid">
                    <div class="rpa-feature"><div class="t">Upload Document</div>
                        <div class="d">Add a research paper (PDF), or load the built-in sample paper to try the app immediately.</div></div>
                    <div class="rpa-feature"><div class="t">Summary</div>
                        <div class="d">Brief, detailed, bullet-point, or structured academic summaries, generated hierarchically so long papers never overflow the model's context window.</div></div>
                    <div class="rpa-feature"><div class="t">Key Information</div>
                        <div class="d">Title, authors, methodology, dataset, results, limitations and more, extracted as structured fields. Missing fields are labelled, never guessed.</div></div>
                    <div class="rpa-feature"><div class="t">Ask Document</div>
                        <div class="d">Questions answered only from the paper's retrieved excerpts and its summary, with the source page shown. Unrelated questions are refused rather than answered from general knowledge.</div></div>
                    <div class="rpa-feature"><div class="t">Evaluation</div>
                        <div class="d">ROUGE and BERTScore for summaries; Precision, Recall and Hit@K for retrieval, each with a chart.</div></div>
                    <div class="rpa-feature"><div class="t">History</div>
                        <div class="d">Revisit any previously uploaded paper and its conversation, or delete it.</div></div>
                </div>
                <p class="rpa-section-sub" style="margin-top:16px;">
                    Embeddings: <code>{_esc(cfg.embedding_model)}</code> &middot;
                    Vector store: <code>{'FAISS' if faiss else 'NumPy (FAISS not installed)'}</code>
                </p>
                {'' if llm.generative else
                 '<p class="rpa-section-sub"><b>Extractive mode:</b> no LLM key is configured, so answers and summaries are built '
                 'from the most relevant sentences of the paper instead of being written by a model. Set <code>GEMINI_API_KEY</code> '
                 '(free, from aistudio.google.com/apikey) before running Section 1 for full LLM answers.</p>'}
                """)

            # ---------------- Upload ----------------
            with gr.Tab("Upload Document"):
                gr.HTML('<p class="rpa-section-title">Upload a research paper</p>'
                       f'<p class="rpa-section-sub">PDF only, up to {cfg.max_upload_mb} MB. Processing runs through validation, '
                       'text extraction, chunking and embedding automatically.</p>')
                file_in = gr.File(label="Research paper (PDF)", file_types=[".pdf"], type="filepath")
                with gr.Row():
                    up_btn = gr.Button("Upload and process", variant="primary")
                    sample_btn = gr.Button("Use the sample paper")
                up_steps = gr.HTML(render_steps(0))
                up_info = gr.HTML()

            # ---------------- Summary ----------------
            with gr.Tab("Summary"):
                gr.HTML('<p class="rpa-section-title">Generate a summary</p>')
                with gr.Row():
                    kind_in = gr.Radio(SummarizationService.TYPES, value="Brief", label="Summary type")
                    inc_info = gr.Checkbox(True, label="Include research information")
                sum_btn = gr.Button("Generate summary", variant="primary")
                sum_out = gr.Markdown()
                sum_file = gr.File(label="Download summary")

            # ---------------- Key Information ----------------
            with gr.Tab("Key Information"):
                gr.HTML('<p class="rpa-section-title">Structured key information</p>'
                       '<p class="rpa-section-sub">Fields not stated in the paper are labelled "Not found in document" rather than guessed.</p>')
                with gr.Row():
                    ki_btn = gr.Button("Extract key information", variant="primary")
                    ki_force = gr.Checkbox(False, label="Re-extract (ignore saved result)")
                ki_out = gr.HTML()
                with gr.Accordion("Structured JSON", open=False):
                    ki_json = gr.JSON()
                ki_file = gr.File(label="Download JSON")

            # ---------------- Ask Document ----------------
            with gr.Tab("Ask Document"):
                gr.HTML('<p class="rpa-section-title">Ask the document</p>'
                       '<p class="rpa-section-sub">Answers are grounded in the paper\'s summary and the most relevant excerpts. '
                       'A question the paper doesn\'t address is refused, not guessed.</p>')
                with gr.Row():
                    with gr.Column(scale=3):
                        chatbot = gr.Chatbot(label=None, type="messages", height=420, show_label=False,
                                             avatar_images=None, placeholder="Ask a question about the uploaded paper to begin.")
                        q_in = gr.Textbox(label="Your question", placeholder="What dataset was used?", lines=1)
                        with gr.Row():
                            ask_btn = gr.Button("Ask", variant="primary")
                            clear_btn = gr.Button("Clear chat")
                        with gr.Accordion("Retrieval settings", open=False):
                            topk_in = gr.Slider(1, 10, value=cfg.top_k, step=1, label="Chunks to retrieve (top K)")
                            thr_in = gr.Slider(0.0, 0.8, value=cfg.relevance_threshold, step=0.01,
                                               label="Relevance threshold (raise it to refuse more unrelated questions)")
                    with gr.Column(scale=2):
                        gr.HTML('<p class="rpa-section-title" style="font-size:1.05rem;">Sources</p>')
                        src_html = gr.HTML()
                        with gr.Row():
                            page_in = gr.Number(value=1, precision=0, label="Open PDF page")
                            page_btn = gr.Button("Show page")
                        page_img = gr.Image(label="Source page", type="pil", height=360)

            # ---------------- Evaluation ----------------
            with gr.Tab("Evaluation"):
                gr.HTML('<p class="rpa-section-title">Summary quality</p>'
                       '<p class="rpa-section-sub">Paste a human-written reference summary; the app generates a fresh summary and scores it with ROUGE and BERTScore.</p>')
                with gr.Row():
                    ev_kind = gr.Dropdown(SummarizationService.TYPES, value="Brief", label="Summary type to evaluate")
                ev_ref = gr.Textbox(label="Reference summary", lines=5)
                ev_btn = gr.Button("Evaluate summary", variant="primary")
                with gr.Row():
                    ev_gen = gr.Textbox(label="Generated summary", lines=5, interactive=False)
                    ev_ref_show = gr.Textbox(label="Reference summary (as used)", lines=5, interactive=False)
                ev_scores = gr.HTML()
                ev_plot = gr.Plot()
                gr.HTML('<p class="rpa-section-title">Retrieval quality</p>'
                       '<p class="rpa-section-sub">One question per line as <code>question | expected pages</code>. '
                       'Leave empty to use the demo questions with keyword-based (silver) page labels.</p>')
                rv_data = gr.Textbox(label="Evaluation dataset", lines=6, placeholder="What dataset was used? | 2")
                with gr.Row():
                    rv_k = gr.Slider(1, 10, value=3, step=1, label="K")
                    rv_fill = gr.Button("Fill with demo questions")
                    rv_btn = gr.Button("Evaluate retrieval", variant="primary")
                rv_out = gr.HTML()
                rv_plot = gr.Plot()

            # ---------------- History ----------------
            with gr.Tab("History"):
                gr.HTML('<p class="rpa-section-title">Previously uploaded documents</p>')
                hist_dd = gr.Dropdown(choices=doc_choices(), label="Uploaded documents", value=None)
                with gr.Row():
                    h_load = gr.Button("Open document", variant="primary")
                    h_refresh = gr.Button("Refresh list")
                    h_delete = gr.Button("Delete document", variant="stop")
                h_msg = gr.HTML()
                h_info = gr.HTML()
                gr.HTML('<p class="rpa-section-title" style="font-size:1.05rem;">Conversation</p>')
                h_chat = gr.Chatbot(label=None, type="messages", height=320, show_label=False)

        # ---- wiring ----
        up_btn.click(cb_upload, file_in, [up_steps, up_info, status_strip, hist_dd])
        sample_btn.click(cb_sample, None, [up_steps, up_info, status_strip, hist_dd])
        sum_btn.click(cb_summary, [kind_in, inc_info], [sum_out, sum_file, status_strip])
        ki_btn.click(cb_keyinfo, ki_force, [ki_out, ki_json, ki_file, status_strip])
        ask_in = [q_in, chatbot, topk_in, thr_in]
        ask_out = [q_in, chatbot, src_html, page_img, status_strip]
        ask_btn.click(cb_ask, ask_in, ask_out)
        q_in.submit(cb_ask, ask_in, ask_out)
        clear_btn.click(cb_clear_chat, None, [chatbot, q_in, src_html, page_img])
        page_btn.click(cb_view_page, page_in, page_img)
        ev_btn.click(cb_eval_summary, [ev_kind, ev_ref], [ev_gen, ev_ref_show, ev_scores, ev_plot])
        rv_fill.click(cb_default_dataset, None, rv_data)
        rv_btn.click(cb_eval_retrieval, [rv_data, rv_k], [rv_out, rv_plot])
        h_load.click(cb_history_load, hist_dd, [status_strip, h_chat, h_info])
        h_refresh.click(cb_refresh, None, hist_dd)
        h_delete.click(cb_history_delete, hist_dd, [status_strip, hist_dd, h_msg, h_info])
    return demo


try:
    demo.close()                      # re-running this cell: close the previous server first
except Exception:
    pass
demo = build_ui()
demo.queue()

import sys
IN_COLAB = "google.colab" in sys.modules
# Colab: the localhost proxy link (…prod.colab.dev) often shows "Connection errored out. Failed to fetch".
# A public gradio.live link (share=True) is far more reliable, and debug=True keeps this cell alive so the server keeps running.
demo.launch(share=SHARE_PUBLIC_LINK or IN_COLAB, inline=True, debug=IN_COLAB)

## 11. Using the system without the web interface
Everything above is plain Python, so you can also drive it from cells (useful for testing or for your viva):

In [ ]:
# doc_id = pipeline.process_file("your_paper.pdf")                 # 1. upload + process
# print(summarizer.summarize(doc_id, "Academic Summary")["text"])   # 2. summaries: Brief | Detailed | Bullet Points | Academic Summary
# print(json.dumps(keyinfo.extract(doc_id), indent=2))              # 3. structured key information
# r = qa.ask(doc_id, "What dataset was used?")                      # 4. grounded question answering
# print(r["answer"], [s["page"] for s in r["sources"]])
# r = qa.ask(doc_id, "How many categories does it contain?")        # 5. follow-up (pronoun resolved automatically)
# print(qa.ask(doc_id, "Who is the Prime Minister of India?")["answer"])  # 6. out-of-context question is refused

## 12. Notes for the viva
* **Why RAG?** The LLM only sees the few chunks most similar to the question, so answers stay grounded in the paper, are cheaper, and can cite pages.
* **Why chunk overlap?** Sentences at chunk borders would otherwise lose context.
* **Cosine similarity:** vectors are L2-normalised, so the FAISS inner product equals cosine similarity.
* **Out-of-context detection:** the best retrieval score is compared with a configurable threshold; below it the LLM is never called.
* **Hierarchical summarization:** long papers are condensed in levels so no single LLM call exceeds the context window.
* **Swappable LLM:** `LLMService` wraps Gemini, OpenAI-compatible APIs, local Hugging Face models, or extractive mode; the rest of the code never changes.
* **Busy-model countermeasures (Gemini):** each model gets a short timeout (default 3.5s); on a timeout, rate limit (429) or server error (503/500) the next model in the list is tried immediately, Flash models first. If every listed model name 404s, the provider asks the API for its current model list and retries a couple of live Flash models, so the notebook keeps working even if Google renames a model.
* **Why give the LLM a summary as well as retrieved chunks?** The top-K chunks are precise but narrow; a short whole-paper summary gives the model orientation (what the paper is about) so it answers more sensibly even when the retrieved excerpts alone are ambiguous.
* **Known limitations:** scanned PDFs need OCR (`ENABLE_OCR=1` + Tesseract); two-column layouts can interleave text; extractive mode is weaker than an LLM; the app keeps one active document at a time (single-user demo).